# Inspect the existing project


In [ ]:

# 1. Mount Google Drive and configure project root
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys
import subprocess

REPO_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

RAW_DIR = REPO_ROOT / "data" / "raw" / "olist"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
SRC_DIR = REPO_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


# 2. Verify project setup
print("1. PROJECT PATH VERIFICATION")

print("Repository exists:", REPO_ROOT.exists())
print("Raw data directory exists:", RAW_DIR.exists())
print("Processed data directory exists:", PROCESSED_DIR.exists())
print("Source directory exists:", SRC_DIR.exists())

assert REPO_ROOT.is_dir(), (
    f"Repository not found: {REPO_ROOT}"
)

assert (REPO_ROOT / ".git").exists(), (
    f"Git metadata not found in: {REPO_ROOT}"
)


# 3. Git inspection helper
def run_git(*args):
    result = subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args],
        capture_output=True,
        text=True
    )

    print(f"\n$ git {' '.join(args)}")
    print(result.stdout or "(no output)")

    if result.stderr:
        print("STDERR:", result.stderr)

    if result.returncode != 0:
        print("Exit code:", result.returncode)

    return result


# 4. Check Git status
print("2. GIT STATUS")

run_git("status", "--short", "--branch")


# 5. Check branch and recent commits
print("3. BRANCH AND RECENT COMMITS")

run_git("branch", "--show-current")
run_git("log", "-5", "--oneline")


# 6. Inspect repository structure
print("4. REPOSITORY STRUCTURE")

for item in sorted(
    REPO_ROOT.iterdir(),
    key=lambda p: (p.is_file(), p.name.lower())
):
    if item.name == ".git":
        continue

    suffix = "/" if item.is_dir() else ""
    print(f"{item.name}{suffix}")


# 7. Inspect project files up to three directory levels
print("5. PROJECT FILES (UP TO 3 LEVELS)")


EXCLUDED_DIRS = {
    ".git",
    "__pycache__",
    ".ipynb_checkpoints"
}

for path in sorted(REPO_ROOT.rglob("*")):
    relative_path = path.relative_to(REPO_ROOT)

    if any(part in EXCLUDED_DIRS for part in relative_path.parts):
        continue

    if len(relative_path.parts) > 3:
        continue

    if path.is_file():
        print(relative_path)


# 8. Inspect raw and processed datasets
print("6. EXISTING DATASETS")


for label, directory in [
    ("RAW OLIST DATA", RAW_DIR),
    ("PROCESSED DATA", PROCESSED_DIR),
]:
    print(f"\n{label}: {directory}")

    if not directory.is_dir():
        print("Directory not found.")
        continue

    files = sorted(
        p for p in directory.rglob("*")
        if p.is_file()
        and not any(
            part in EXCLUDED_DIRS
            for part in p.relative_to(directory).parts
        )
    )

    if not files:
        print("No files found.")
        continue

    for file_path in files:
        size_mb = file_path.stat().st_size / (1024 * 1024)
        print(
            f"{file_path.relative_to(directory)}"
            f" | {size_mb:.2f} MB"
        )

    print("Total files:", len(files))


# 9. Inspect SQL analytics files

print("7. SQL ANALYTICS FILES")


SQL_DIR = REPO_ROOT / "sql"

if SQL_DIR.is_dir():
    sql_files = sorted(SQL_DIR.rglob("*.sql"))

    for sql_file in sql_files:
        print(sql_file.relative_to(REPO_ROOT))

    print(f"\nTotal SQL files found: {len(sql_files)}")
else:
    print("SQL directory does not exist.")


# 10. Locate relevant Phase 1–3 artifacts
print("8. PHASE 1–3 ARTIFACTS")

RELEVANT_PATTERNS = [
    "*pipeline*.py",
    "*transform*.py",
    "*test*.py",
    "*phase-*.md",
    "*requirements*.txt",
    "*PySpark*.ipynb",
    "*postgresql*.ipynb",
    "*warehouse*.ipynb",
    "*.sql"
]

seen = set()

for pattern in RELEVANT_PATTERNS:
    for path in sorted(REPO_ROOT.rglob(pattern)):
        relative_path = path.relative_to(REPO_ROOT)

        if any(part in EXCLUDED_DIRS for part in relative_path.parts):
            continue

        if relative_path in seen:
            continue

        seen.add(relative_path)
        print(relative_path)


# 11. Recent commit summaries

print("9. RECENT COMMIT DETAILS")


run_git("log", "-3", "--stat", "--oneline")

print("REPOSITORY INSPECTION FINISHED")
print("No project files were modified by this inspection cell.")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1. PROJECT PATH VERIFICATION
Repository exists: True
Raw data directory exists: True
Processed data directory exists: True
Source directory exists: True
2. GIT STATUS

$ git status --short --branch
## main...origin/main
 M 02_postgresql_data_warehouse.ipynb
 M 03_PySpark_transformation.ipynb
?? 04_Storage_and_Data_modeling.ipynb

3. BRANCH AND RECENT COMMITS

$ git branch --show-current
main


$ git log -5 --oneline
ec52c9a Merge remote-tracking branch 'origin/main'
9bd4599 feat: add tested PySpark transformations
f4a2643 Add SQL analytics queries
a99d9c4 Add data ingestion and PostgreSQL warehouse notebooks
eb48739 Add data ingestion and PostgreSQL warehouse notebooks

4. REPOSITORY STRUCTURE
config/
data/
docs/
logs/
notebooks/
reports/
sql/
src/
tests/
.gitignore
00_phase0_setup.ipynb
01_data_ingestion.ipynb
02_postgresql_data_warehouse.ipynb
03_PySpark_tr

Inspect the schemas and existing logic

In [ ]:

from pathlib import Path
import ast
import json
import re
import pandas as pd

print("1. CSV SCHEMAS")

for directory in [RAW_DIR, PROCESSED_DIR]:
    if not directory.is_dir():
        continue

    print(f"\n{directory.relative_to(REPO_ROOT)}")

    for path in sorted(directory.glob("*.csv")):
        try:
            columns = pd.read_csv(path, nrows=0).columns.tolist()
            print(f"\n{path.name} ({len(columns)} columns)")
            print(", ".join(columns))
        except Exception as exc:
            print(f"{path.name}: inspection failed ({type(exc).__name__})")


print("\n2. PYTHON MODULES AND FUNCTION SIGNATURES")

MODULES = [
    SRC_DIR / "sales_pipeline" / "pipeline.py",
    SRC_DIR / "sales_pipeline" / "transform.py",
    SRC_DIR / "sales_pipeline" / "transform_sales.py",
    SRC_DIR / "sales_pipeline" / "transform_customers.py",
    SRC_DIR / "sales_pipeline" / "transform_products.py",
    SRC_DIR / "sales_pipeline" / "validate.py",
    SRC_DIR / "sales_pipeline" / "validation.py",
    SRC_DIR / "sales_pipeline" / "spark_transformations.py",
]

for path in MODULES:
    if not path.is_file():
        continue

    try:
        tree = ast.parse(path.read_text(encoding="utf-8"))
        functions = [
            node.name
            for node in ast.walk(tree)
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
        ]
        classes = [
            node.name for node in tree.body
            if isinstance(node, ast.ClassDef)
        ]

        print(f"\n{path.relative_to(REPO_ROOT)}")
        print("Functions:", ", ".join(functions) or "None")
        print("Classes:", ", ".join(classes) or "None")

    except Exception as exc:
        print(f"{path.name}: inspection failed ({type(exc).__name__})")


print("\n3. NOTEBOOK HEADINGS")

NOTEBOOKS = [
    REPO_ROOT / "01_data_ingestion.ipynb",
    REPO_ROOT / "02_postgresql_data_warehouse.ipynb",
    REPO_ROOT / "03_PySpark_transformation.ipynb",
    REPO_ROOT / "04_Storage_and_Data_modeling.ipynb",
]

for path in NOTEBOOKS:
    if not path.is_file():
        continue

    try:
        notebook = json.loads(path.read_text(encoding="utf-8"))
        headings = []

        for cell in notebook.get("cells", []):
            source = "".join(cell.get("source", []))

            if cell.get("cell_type") == "markdown":
                for line in source.splitlines():
                    if re.match(r"^\s{0,3}#{1,3}\s+", line):
                        headings.append(line.strip())

        print(f"\n{path.name}")
        print("\n".join(headings[:30]) or "No Markdown headings")

    except Exception as exc:
        print(f"{path.name}: inspection failed ({type(exc).__name__})")


print("\n4. EXISTING SQL TABLE DEFINITIONS")

SQL_DIR = REPO_ROOT / "sql"
found_ddl = False

if SQL_DIR.is_dir():
    for path in sorted(SQL_DIR.rglob("*.sql")):
        text = path.read_text(encoding="utf-8")

        matches = re.findall(
            r"\b(?:CREATE\s+TABLE|CREATE\s+VIEW)\b[^;]*;",
            text,
            flags=re.IGNORECASE | re.DOTALL
        )

        if matches:
            found_ddl = True
            print(f"\n{path.relative_to(REPO_ROOT)}")
            for statement in matches:
                print(" ".join(statement.split())[:700])

if not found_ddl:
    print("No CREATE TABLE or CREATE VIEW statements found in SQL files.")
    print("Warehouse table definitions may be inside the Phase 2 notebook.")


print("\nInspection complete. No project files were modified.")


1. CSV SCHEMAS

data/raw/olist

olist_customers_dataset.csv (5 columns)
customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state

olist_geolocation_dataset.csv (5 columns)
geolocation_zip_code_prefix, geolocation_lat, geolocation_lng, geolocation_city, geolocation_state

olist_order_items_dataset.csv (7 columns)
order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value

olist_order_payments_dataset.csv (5 columns)
order_id, payment_sequential, payment_type, payment_installments, payment_value

olist_order_reviews_dataset.csv (7 columns)
review_id, order_id, review_score, review_comment_title, review_comment_message, review_creation_date, review_answer_timestamp

olist_orders_dataset.csv (8 columns)
order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date

olist_products_dataset.csv (9 columns)
product_id, pro

Inspect the Existing transformation and warehouse definations

In [ ]:

import ast
import json
import re

print("1. EXISTING PYTHON TRANSFORMATION LOGIC")

FUNCTIONS_TO_INSPECT = {
    "transform.py": [
        "build_order_analytics",
    ],
    "transform_sales.py": [
        "build_sales_analytics",
    ],
    "transform_customers.py": [
        "build_customer_analytics",
    ],
    "transform_products.py": [
        "build_product_analytics",
    ],
    "validate.py": [
        "validate_data",
        "_check_unique",
        "_check_foreign_key",
        "_check_non_negative",
    ],
    "spark_transformations.py": [
        "build_order_analytics_spark",
        "prepare_sales_items_spark",
        "build_product_analytics_spark",
        "build_category_analytics_spark",
    ],
}

SENSITIVE_PATTERN = re.compile(
    r"password|secret|NEON_DB_URL|connection.?string|"
    r"credential|api.?key|access.?token",
    re.IGNORECASE,
)

def redact_line(line):
    if SENSITIVE_PATTERN.search(line):
        return "[REDACTED: potentially sensitive line]"
    return line

for filename, selected_functions in FUNCTIONS_TO_INSPECT.items():
    path = REPO_ROOT / "src" / "sales_pipeline" / filename

    if not path.is_file():
        continue

    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source)
    function_nodes = {
        node.name: node
        for node in ast.walk(tree)
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
    }

    print(f"\n{path.relative_to(REPO_ROOT)}")

    for function_name in selected_functions:
        node = function_nodes.get(function_name)

        if node is None:
            print(f"\n{function_name}: not found")
            continue

        print(f"\n--- {function_name} ---")
        lines = source.splitlines()

        for line in lines[node.lineno - 1:node.end_lineno]:
            print(redact_line(line))


print("\n2. PHASE 2 POSTGRESQL SCHEMA AND LOADING LOGIC")

notebook_path = REPO_ROOT / "02_postgresql_data_warehouse.ipynb"

if notebook_path.is_file():
    notebook = json.loads(
        notebook_path.read_text(encoding="utf-8")
    )

    keywords = re.compile(
        r"CREATE TABLE|CREATE SCHEMA|CREATE INDEX|"
        r"to_sql|INSERT INTO|executemany|"
        r"information_schema|PRIMARY KEY|FOREIGN KEY|"
        r"CREATE TABLE IF NOT EXISTS|"
        r"schema\s*=|dtype\s*=|chunksize|"
        r"TRUNCATE|DELETE FROM|BEGIN|COMMIT|"
        r"order_analytics|customer_analytics|"
        r"product_analytics|category_analytics|"
        r"monthly_sales_analytics|order_status_summary",
        re.IGNORECASE,
    )

    for index, cell in enumerate(notebook.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        lines = source.splitlines()

        matched = [
            i for i, line in enumerate(lines)
            if keywords.search(line)
        ]

        if not matched:
            continue

        print(f"\nPhase 2 code cell {index}")

        # Show matched lines with nearby context, without dumping
        # unrelated code or notebook outputs.
        selected_lines = set()

        for i in matched:
            selected_lines.update(
                range(max(0, i - 2), min(len(lines), i + 4))
            )

        previous = -2

        for i in sorted(selected_lines):
            if i > previous + 1:
                print("...")

            print(redact_line(lines[i]))
            previous = i

else:
    print("Phase 2 notebook not found.")


print("\nInspection complete. No notebook or source files modified.")


1. EXISTING PYTHON TRANSFORMATION LOGIC

src/sales_pipeline/transform.py

--- build_order_analytics ---
def build_order_analytics(data_dir):
    """
    Build one row per order.

    Revenue is calculated from order items; payment totals are aggregated
    independently to avoid multiplying values during joins.
    """
    data_dir = Path(data_dir)

    orders = load_csv(data_dir, "orders")
    items = load_csv(data_dir, "order_items")
    payments = load_csv(data_dir, "payments")

    # Parse timestamps; invalid or missing values remain NaT.
    date_columns = [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ]

    for column in date_columns:
        orders[column] = pd.to_datetime(orders[column], errors="coerce")

    # Aggregate item-level data to one row per order.
    item_metrics = (
        items.groupby("order_id", as_index=False)
     

Define the dimensional model


In [7]:

import pandas as pd

print("Source key integrity checks")

customers = pd.read_csv(RAW_DIR / "olist_customers_dataset.csv")
orders = pd.read_csv(RAW_DIR / "olist_orders_dataset.csv")
items = pd.read_csv(RAW_DIR / "olist_order_items_dataset.csv")
products = pd.read_csv(RAW_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW_DIR / "olist_sellers_dataset.csv")

checks = [
    ("customers", customers, ["customer_id"]),
    ("orders", orders, ["order_id"]),
    ("order items", items, ["order_id", "order_item_id"]),
    ("products", products, ["product_id"]),
    ("sellers", sellers, ["seller_id"]),
]

for name, df, keys in checks:
    null_rows = int(df[keys].isna().any(axis=1).sum())
    duplicate_rows = int(df.duplicated(subset=keys).sum())

    print(
        f"{name}: rows={len(df):,}, "
        f"null_key_rows={null_rows:,}, "
        f"duplicate_key_rows={duplicate_rows:,}, "
        f"key={'+'.join(keys)}"
    )

print("\nForeign-key integrity checks")

relationships = [
    ("orders.customer_id", orders["customer_id"],
     customers["customer_id"]),
    ("items.order_id", items["order_id"],
     orders["order_id"]),
    ("items.product_id", items["product_id"],
     products["product_id"]),
    ("items.seller_id", items["seller_id"],
     sellers["seller_id"]),
]

for name, child, parent in relationships:
    missing = int((child.notna() & ~child.isin(parent)).sum())
    nulls = int(child.isna().sum())

    print(
        f"{name}: unmatched={missing:,}, nulls={nulls:,}"
    )

print("\nOrder status distribution")
print(orders["order_status"].value_counts(dropna=False).to_string())

print("\nChecks finished. No files or database tables were modified.")


Source key integrity checks
customers: rows=99,441, null_key_rows=0, duplicate_key_rows=0, key=customer_id
orders: rows=99,441, null_key_rows=0, duplicate_key_rows=0, key=order_id
order items: rows=112,650, null_key_rows=0, duplicate_key_rows=0, key=order_id+order_item_id
products: rows=32,951, null_key_rows=0, duplicate_key_rows=0, key=product_id
sellers: rows=3,095, null_key_rows=0, duplicate_key_rows=0, key=seller_id

Foreign-key integrity checks
orders.customer_id: unmatched=0, nulls=0
items.order_id: unmatched=0, nulls=0
items.product_id: unmatched=0, nulls=0
items.seller_id: unmatched=0, nulls=0

Order status distribution
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2

Checks finished. No files or database tables were modified.


Confim customer identity and sales eligibility

In [8]:

import ast

print("1. CUSTOMER IDENTITY")

customers = pd.read_csv(RAW_DIR / "olist_customers_dataset.csv")

print("Customer records:", len(customers))
print("Unique customer_id:", customers["customer_id"].nunique())
print("Unique customer_unique_id:", customers["customer_unique_id"].nunique())
print(
    "Repeated customer_unique_id records:",
    int(customers["customer_unique_id"].duplicated().sum())
)

print("\nCustomer attributes across repeated identities")

identity_counts = (
    customers.groupby("customer_unique_id")
    .agg(
        customer_records=("customer_id", "nunique"),
        distinct_states=("customer_state", "nunique"),
        distinct_cities=("customer_city", "nunique")
    )
)

repeated = identity_counts[identity_counts["customer_records"] > 1]

print("Identities with multiple customer records:", len(repeated))
print(
    "Repeated identities with multiple states:",
    int((repeated["distinct_states"] > 1).sum())
)
print(
    "Repeated identities with multiple cities:",
    int((repeated["distinct_cities"] > 1).sum())
)


print("\n2. EXISTING SALES EXCLUSION RULE")

source_files = [
    SRC_DIR / "sales_pipeline" / "transform_sales.py",
    SRC_DIR / "sales_pipeline" / "transform_customers.py",
    SRC_DIR / "sales_pipeline" / "transform_products.py",
    SRC_DIR / "sales_pipeline" / "spark_transformations.py",
]

for path in source_files:
    if not path.is_file():
        continue

    tree = ast.parse(path.read_text(encoding="utf-8"))
    found = False

    for node in ast.walk(tree):
        if isinstance(node, ast.Assign):
            names = [
                target.id
                for target in node.targets
                if isinstance(target, ast.Name)
            ]

            if "EXCLUDED_SALES_STATUSES" in names:
                print(f"{path.name}:")
                print(ast.unparse(node))
                found = True
                break

    if found:
        break
else:
    print("Exclusion constant not found in inspected modules.")


print("\nInspection complete. No files were modified.")


1. CUSTOMER IDENTITY
Customer records: 99441
Unique customer_id: 99441
Unique customer_unique_id: 96096
Repeated customer_unique_id records: 3345

Customer attributes across repeated identities
Identities with multiple customer records: 2997
Repeated identities with multiple states: 39
Repeated identities with multiple cities: 122

2. EXISTING SALES EXCLUSION RULE
transform_sales.py:
EXCLUDED_SALES_STATUSES = {'canceled', 'unavailable'}

Inspection complete. No files were modified.


# Implement the Storage layout

Create the storage directories

In [9]:

from pathlib import Path

# Define Phase 4 storage locations.
DATA_DIR = REPO_ROOT / "data"

BRONZE_DIR = DATA_DIR / "bronze" / "olist"

SILVER_DIR = DATA_DIR / "silver"

SILVER_DATASETS = [
    "customers",
    "orders",
    "order_items",
    "payments",
    "products",
    "sellers",
]

GOLD_ANALYTICS_DIR = DATA_DIR / "gold" / "analytics"

GOLD_DIMENSIONAL_DIR = DATA_DIR / "gold" / "dimensional"

DIMENSIONAL_TABLES = [
    "fact_sales",
    "dim_customer",
    "dim_product",
    "dim_seller",
    "dim_order",
]

# Create directories without deleting or overwriting existing files.
directories = [
    BRONZE_DIR,
    GOLD_ANALYTICS_DIR,
]

directories.extend(
    SILVER_DIR / dataset
    for dataset in SILVER_DATASETS
)

directories.extend(
    GOLD_DIMENSIONAL_DIR / table
    for table in DIMENSIONAL_TABLES
)

for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)

print("Phase 4 storage directories")

for directory in directories:
    relative_path = directory.relative_to(REPO_ROOT)
    print(f"[OK] {relative_path}")

# Verify all directories exist.
missing = [
    directory
    for directory in directories
    if not directory.is_dir()
]

assert not missing, f"Missing directories: {missing}"

print(f"\nVerified: {len(directories)} directories created or already present.")
print("Existing source and processed datasets were not modified.")


Phase 4 storage directories
[OK] data/bronze/olist
[OK] data/gold/analytics
[OK] data/silver/customers
[OK] data/silver/orders
[OK] data/silver/order_items
[OK] data/silver/payments
[OK] data/silver/products
[OK] data/silver/sellers
[OK] data/gold/dimensional/fact_sales
[OK] data/gold/dimensional/dim_customer
[OK] data/gold/dimensional/dim_product
[OK] data/gold/dimensional/dim_seller
[OK] data/gold/dimensional/dim_order

Verified: 13 directories created or already present.
Existing source and processed datasets were not modified.


Inspect *.gitignore*

In [10]:

GITIGNORE_PATH = REPO_ROOT / ".gitignore"

print("Current .gitignore")

if GITIGNORE_PATH.is_file():
    print(GITIGNORE_PATH.read_text(encoding="utf-8"))
else:
    print(".gitignore does not exist.")

print("\nCurrent Git status")

result = subprocess.run(
    ["git", "-C", str(REPO_ROOT), "status", "--short"],
    capture_output=True,
    text=True,
    check=True,
)

print(result.stdout or "No changes reported.")


Current .gitignore
# Python
__pycache__/
*.py[cod]
.venv/
venv/
.env

# Jupyter
.ipynb_checkpoints/

# Local data (keep real datasets out of Git)
data/*
!data/.gitkeep

# Generated logs and outputs
logs/*
!logs/.gitkeep
outputs/

# Reports: commit only if intentionally version-controlled
reports/*
!reports/.gitkeep

# Local database files
*.sqlite
*.db


Current Git status
 M 02_postgresql_data_warehouse.ipynb
 M 03_PySpark_transformation.ipynb
?? 04_Storage_and_Data_modeling.ipynb



Varify the Spark environment and Bronze destination

In [11]:

import importlib.util
from pathlib import Path

print("Spark environment")

pyspark_available = importlib.util.find_spec("pyspark") is not None
print("PySpark installed:", pyspark_available)

if pyspark_available:
    import pyspark
    print("PySpark version:", pyspark.__version__)

print("\nBronze source and destination")

print("Raw CSV directory exists:", RAW_DIR.is_dir())
print("Bronze directory exists:", BRONZE_DIR.is_dir())

raw_files = sorted(RAW_DIR.glob("*.csv")) if RAW_DIR.is_dir() else []

print("Source CSV files:", len(raw_files))
for path in raw_files:
    print(f"  {path.name}")

existing_bronze_files = (
    sorted(p for p in BRONZE_DIR.rglob("*") if p.is_file())
    if BRONZE_DIR.is_dir()
    else []
)

print("\nExisting Bronze files:", len(existing_bronze_files))
for path in existing_bronze_files[:30]:
    print(f"  {path.relative_to(REPO_ROOT)}")

if len(existing_bronze_files) > 30:
    print(f"  ... and {len(existing_bronze_files) - 30} more")

print("\nEnvironment inspection complete. No files were modified.")


Spark environment
PySpark installed: True
PySpark version: 4.0.4

Bronze source and destination
Raw CSV directory exists: True
Bronze directory exists: True
Source CSV files: 9
  olist_customers_dataset.csv
  olist_geolocation_dataset.csv
  olist_order_items_dataset.csv
  olist_order_payments_dataset.csv
  olist_order_reviews_dataset.csv
  olist_orders_dataset.csv
  olist_products_dataset.csv
  olist_sellers_dataset.csv
  product_category_name_translation.csv

Existing Bronze files: 0

Environment inspection complete. No files were modified.


Writes raw dataset to bronze in parquet

In [12]:

from pathlib import Path

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SalesPipeline-Phase4-Bronze")
    .master("local[*]")
    .getOrCreate()
)

BRONZE_DIR.mkdir(parents=True, exist_ok=True)

raw_csv_files = sorted(RAW_DIR.glob("*.csv"))

if len(raw_csv_files) != 9:
    raise RuntimeError(
        f"Expected 9 raw CSV files, found {len(raw_csv_files)}."
    )

# Preflight all destinations before writing anything.
bronze_jobs = []

for csv_path in raw_csv_files:
    dataset_name = csv_path.stem
    output_path = BRONZE_DIR / dataset_name

    if output_path.exists() and any(output_path.iterdir()):
        raise FileExistsError(
            f"Refusing to overwrite existing Bronze data: {output_path}"
        )

    bronze_jobs.append((csv_path, output_path))

# Read and persist each source dataset.
bronze_results = []

for csv_path, output_path in bronze_jobs:
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(str(csv_path))
    )

    source_count = df.count()

    df.write \
        .mode("errorifexists") \
        .parquet(str(output_path))

    reloaded = spark.read.parquet(str(output_path))
    saved_count = reloaded.count()

    if source_count != saved_count:
        raise AssertionError(
            f"Row-count mismatch for {csv_path.name}: "
            f"source={source_count}, saved={saved_count}"
        )

    bronze_results.append({
        "dataset": csv_path.name,
        "rows": saved_count,
        "columns": len(reloaded.columns),
        "status": "PASS",
    })

    print(
        f"{csv_path.name}: {saved_count:,} rows, "
        f"{len(reloaded.columns)} columns — PASS"
    )

print(f"\nBronze datasets persisted and reloaded: {len(bronze_results)}")


olist_customers_dataset.csv: 99,441 rows, 5 columns — PASS
olist_geolocation_dataset.csv: 1,000,163 rows, 5 columns — PASS
olist_order_items_dataset.csv: 112,650 rows, 7 columns — PASS
olist_order_payments_dataset.csv: 103,886 rows, 5 columns — PASS
olist_order_reviews_dataset.csv: 104,162 rows, 7 columns — PASS
olist_orders_dataset.csv: 99,441 rows, 8 columns — PASS
olist_products_dataset.csv: 32,951 rows, 9 columns — PASS
olist_sellers_dataset.csv: 3,095 rows, 4 columns — PASS
product_category_name_translation.csv: 71 rows, 2 columns — PASS

Bronze datasets persisted and reloaded: 9


In [13]:

from pyspark.sql import functions as F

schema_results = []

for csv_path in sorted(RAW_DIR.glob("*.csv")):
    dataset_name = csv_path.stem
    parquet_path = BRONZE_DIR / dataset_name

    source_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(str(csv_path))
    )

    saved_df = spark.read.parquet(str(parquet_path))

    source_schema = [
        (field.name, field.dataType.simpleString())
        for field in source_df.schema.fields
    ]

    saved_schema = [
        (field.name, field.dataType.simpleString())
        for field in saved_df.schema.fields
    ]

    source_count = source_df.count()
    saved_count = saved_df.count()

    schema_match = source_schema == saved_schema
    row_count_match = source_count == saved_count

    result = (
        schema_match
        and row_count_match
    )

    schema_results.append({
        "dataset": csv_path.name,
        "source_rows": source_count,
        "parquet_rows": saved_count,
        "schema_match": schema_match,
        "row_count_match": row_count_match,
        "status": "PASS" if result else "FAIL",
    })

    print(
        f"{csv_path.name}: "
        f"schema={'PASS' if schema_match else 'FAIL'}, "
        f"rows={'PASS' if row_count_match else 'FAIL'}"
    )

    if not schema_match:
        print("  Source schema:", source_schema)
        print("  Parquet schema:", saved_schema)

failed = [
    result for result in schema_results
    if result["status"] != "PASS"
]

print(
    f"\nSchema validation: "
    f"{len(schema_results) - len(failed)}/{len(schema_results)} passed"
)

if failed:
    raise AssertionError(
        f"Bronze validation failed for: "
        f"{[result['dataset'] for result in failed]}"
    )

print("All Bronze schemas and row counts match.")


olist_customers_dataset.csv: schema=PASS, rows=PASS
olist_geolocation_dataset.csv: schema=PASS, rows=PASS
olist_order_items_dataset.csv: schema=PASS, rows=PASS
olist_order_payments_dataset.csv: schema=PASS, rows=PASS
olist_order_reviews_dataset.csv: schema=PASS, rows=PASS
olist_orders_dataset.csv: schema=PASS, rows=PASS
olist_products_dataset.csv: schema=PASS, rows=PASS
olist_sellers_dataset.csv: schema=PASS, rows=PASS
product_category_name_translation.csv: schema=PASS, rows=PASS

Schema validation: 9/9 passed
All Bronze schemas and row counts match.


In [14]:

from pyspark.sql import functions as F

print("Bronze data-quality profile")

profile_rules = {
    "olist_customers_dataset": {
        "required": ["customer_id"],
        "non_negative": [],
    },
    "olist_orders_dataset": {
        "required": ["order_id", "customer_id"],
        "non_negative": [],
    },
    "olist_order_items_dataset": {
        "required": ["order_id", "order_item_id", "product_id", "seller_id"],
        "non_negative": ["price", "freight_value"],
    },
    "olist_order_payments_dataset": {
        "required": ["order_id"],
        "non_negative": ["payment_value", "payment_installments"],
    },
    "olist_order_reviews_dataset": {
        "required": ["review_id", "order_id"],
        "non_negative": [],
    },
    "olist_products_dataset": {
        "required": ["product_id"],
        "non_negative": [
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm",
        ],
    },
    "olist_sellers_dataset": {
        "required": ["seller_id"],
        "non_negative": [],
    },
}

for dataset, rules in profile_rules.items():
    path = BRONZE_DIR / dataset
    df = spark.read.parquet(str(path))

    expressions = [
        F.count("*").alias("row_count")
    ]

    for column in rules["required"]:
        expressions.append(
            F.sum(
                F.when(F.col(column).isNull(), 1).otherwise(0)
            ).alias(f"null_{column}")
        )

    for column in rules["non_negative"]:
        expressions.append(
            F.sum(
                F.when(
                    F.col(column).isNotNull() & (F.col(column) < 0),
                    1
                ).otherwise(0)
            ).alias(f"negative_{column}")
        )

    result = df.agg(*expressions).first().asDict()

    print(f"\n{dataset}")
    for metric, value in result.items():
        print(f"  {metric}: {value}")

print("\nProfiling complete. Bronze data remains unchanged.")


Bronze data-quality profile

olist_customers_dataset
  row_count: 99441
  null_customer_id: 0

olist_orders_dataset
  row_count: 99441
  null_order_id: 0
  null_customer_id: 0

olist_order_items_dataset
  row_count: 112650
  null_order_id: 0
  null_order_item_id: 0
  null_product_id: 0
  null_seller_id: 0
  negative_price: 0
  negative_freight_value: 0

olist_order_payments_dataset
  row_count: 103886
  null_order_id: 0
  negative_payment_value: 0
  negative_payment_installments: 0

olist_order_reviews_dataset
  row_count: 104162
  null_review_id: 1
  null_order_id: 2236

olist_products_dataset
  row_count: 32951
  null_product_id: 0
  negative_product_weight_g: 0
  negative_product_length_cm: 0
  negative_product_height_cm: 0
  negative_product_width_cm: 0

olist_sellers_dataset
  row_count: 3095
  null_seller_id: 0

Profiling complete. Bronze data remains unchanged.


Prepare the complete **Silver layer**

In [15]:
from pathlib import Path

SILVER_DIR = Path("data/silver")

silver_datasets = [
    "customers",
    "geolocation",
    "order_items",
    "payments",
    "reviews",
    "orders",
    "products",
    "sellers",
    "category_translation",
]

for dataset in silver_datasets:
    (SILVER_DIR / dataset).mkdir(parents=True, exist_ok=True)

print("Silver dataset directories:")
for dataset in silver_datasets:
    path = SILVER_DIR / dataset
    print(f"{dataset}: {'READY' if path.is_dir() else 'MISSING'}")

Silver dataset directories:
customers: READY
geolocation: READY
order_items: READY
payments: READY
reviews: READY
orders: READY
products: READY
sellers: READY
category_translation: READY


Standardize Silver source datasets

In [19]:
from pyspark.sql import functions as F

bronze_path = BRONZE_DIR / "olist_customers_dataset"
silver_path = SILVER_DIR / "customers"

# Remove the directory only if it is empty.
if silver_path.exists():
    if any(silver_path.iterdir()):
        raise RuntimeError(
            f"{silver_path} is not empty. Refusing to overwrite existing data."
        )
    silver_path.rmdir()

# Read Bronze data.
customers_silver = spark.read.parquet(str(bronze_path))

# Standardize column names.
for column in customers_silver.columns:
    standardized_name = column.strip().lower()
    if column != standardized_name:
        customers_silver = customers_silver.withColumnRenamed(
            column, standardized_name
        )

# Trim string values while preserving nulls.
for column, data_type in customers_silver.dtypes:
    if data_type == "string":
        customers_silver = customers_silver.withColumn(
            column,
            F.trim(F.col(column))
        )

# Persist Silver data.
customers_silver.write.mode("errorifexists").parquet(str(silver_path))

# Reload and validate.
bronze_count = spark.read.parquet(str(bronze_path)).count()
customers_reloaded = spark.read.parquet(str(silver_path))
silver_count = customers_reloaded.count()

print(f"Bronze rows: {bronze_count}")
print(f"Silver rows: {silver_count}")
print(f"Row counts match: {bronze_count == silver_count}")
print("Silver columns:", customers_reloaded.columns)
customers_reloaded.printSchema()

assert bronze_count == silver_count, "Customer row count mismatch"
print("Customer Silver persistence and row-count validation passed.")


Bronze rows: 99441
Silver rows: 99441
Row counts match: True
Silver columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

Customer Silver persistence and row-count validation passed.


In [20]:
from pyspark.sql import functions as F

bronze_path = BRONZE_DIR / "olist_orders_dataset"
silver_path = SILVER_DIR / "orders"

# Protect any existing output.
if silver_path.exists():
    if any(silver_path.iterdir()):
        raise RuntimeError(
            f"{silver_path} is not empty. Refusing to overwrite existing data."
        )
    silver_path.rmdir()

# Read from Bronze.
orders_silver = spark.read.parquet(str(bronze_path))

# Standardize column names.
for column in orders_silver.columns:
    standardized_name = column.strip().lower()
    if column != standardized_name:
        orders_silver = orders_silver.withColumnRenamed(
            column, standardized_name
        )

# Trim string fields while preserving null values.
for column, data_type in orders_silver.dtypes:
    if data_type == "string":
        orders_silver = orders_silver.withColumn(
            column, F.trim(F.col(column))
        )

# Persist Silver.
orders_silver.write.mode("errorifexists").parquet(str(silver_path))

# Reload and validate persistence.
bronze_count = spark.read.parquet(str(bronze_path)).count()
orders_reloaded = spark.read.parquet(str(silver_path))
silver_count = orders_reloaded.count()

print(f"Bronze rows: {bronze_count}")
print(f"Silver rows: {silver_count}")
print(f"Row counts match: {bronze_count == silver_count}")
print("Silver columns:", orders_reloaded.columns)
orders_reloaded.printSchema()

assert bronze_count == silver_count, "Orders row count mismatch"
print("Orders Silver persistence and row-count validation passed.")

Bronze rows: 99441
Silver rows: 99441
Row counts match: True
Silver columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

Orders Silver persistence and row-count validation passed.


In [21]:
from pyspark.sql import functions as F

bronze_path = BRONZE_DIR / "olist_order_items_dataset"
silver_path = SILVER_DIR / "order_items"

# Protect existing data.
if silver_path.exists():
    if any(silver_path.iterdir()):
        raise RuntimeError(
            f"{silver_path} is not empty. Refusing to overwrite existing data."
        )
    silver_path.rmdir()

# Read Bronze.
items_silver = spark.read.parquet(str(bronze_path))

# Standardize column names.
for column in items_silver.columns:
    standardized_name = column.strip().lower()
    if column != standardized_name:
        items_silver = items_silver.withColumnRenamed(
            column, standardized_name
        )

# Trim string values while preserving nulls.
for column, data_type in items_silver.dtypes:
    if data_type == "string":
        items_silver = items_silver.withColumn(
            column, F.trim(F.col(column))
        )

# Persist Silver.
items_silver.write.mode("errorifexists").parquet(str(silver_path))

# Reload and validate.
bronze_count = spark.read.parquet(str(bronze_path)).count()
items_reloaded = spark.read.parquet(str(silver_path))
silver_count = items_reloaded.count()

print(f"Bronze rows: {bronze_count}")
print(f"Silver rows: {silver_count}")
print(f"Row counts match: {bronze_count == silver_count}")
print("Silver columns:", items_reloaded.columns)
items_reloaded.printSchema()

# Check the composite source key.
key_issues = items_reloaded.filter(
    F.col("order_id").isNull()
    | F.col("order_item_id").isNull()
).count()

duplicate_keys = (
    items_reloaded
    .groupBy("order_id", "order_item_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Rows with null composite-key fields: {key_issues}")
print(f"Duplicate composite keys: {duplicate_keys}")

assert bronze_count == silver_count, "Order-items row count mismatch"
assert key_issues == 0, "Null composite-key fields detected"
assert duplicate_keys == 0, "Duplicate composite keys detected"

print("Order-items Silver persistence and key validation passed.")

Bronze rows: 112650
Silver rows: 112650
Row counts match: True
Silver columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

Rows with null composite-key fields: 0
Duplicate composite keys: 0
Order-items Silver persistence and key validation passed.


In [22]:
from pyspark.sql import functions as F

bronze_path = BRONZE_DIR / "olist_order_payments_dataset"
silver_path = SILVER_DIR / "payments"

# Protect existing data.
if silver_path.exists():
    if any(silver_path.iterdir()):
        raise RuntimeError(
            f"{silver_path} is not empty. Refusing to overwrite existing data."
        )
    silver_path.rmdir()

# Read Bronze.
payments_silver = spark.read.parquet(str(bronze_path))

# Standardize column names.
for column in payments_silver.columns:
    standardized_name = column.strip().lower()
    if column != standardized_name:
        payments_silver = payments_silver.withColumnRenamed(
            column, standardized_name
        )

# Trim string values while preserving nulls.
for column, data_type in payments_silver.dtypes:
    if data_type == "string":
        payments_silver = payments_silver.withColumn(
            column, F.trim(F.col(column))
        )

# Persist Silver.
payments_silver.write.mode("errorifexists").parquet(str(silver_path))

# Reload and validate.
bronze_count = spark.read.parquet(str(bronze_path)).count()
payments_reloaded = spark.read.parquet(str(silver_path))
silver_count = payments_reloaded.count()

print(f"Bronze rows: {bronze_count}")
print(f"Silver rows: {silver_count}")
print(f"Row counts match: {bronze_count == silver_count}")
print("Silver columns:", payments_reloaded.columns)
payments_reloaded.printSchema()

# Profile required keys and non-negative measures.
quality = payments_reloaded.agg(
    F.sum(F.when(F.col("order_id").isNull(), 1).otherwise(0))
        .alias("null_order_id"),
    F.sum(F.when(F.col("payment_value") < 0, 1).otherwise(0))
        .alias("negative_payment_value"),
    F.sum(F.when(F.col("payment_installments") < 0, 1).otherwise(0))
        .alias("negative_payment_installments"),
).first().asDict()

print("Quality checks:", quality)

assert bronze_count == silver_count, "Payment row count mismatch"
assert quality["null_order_id"] == 0, "Null order_id values detected"
assert quality["negative_payment_value"] == 0, "Negative payment values detected"
assert quality["negative_payment_installments"] == 0, "Negative installments detected"

print("Payments Silver persistence and quality validation passed.")

Bronze rows: 103886
Silver rows: 103886
Row counts match: True
Silver columns: ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)

Quality checks: {'null_order_id': 0, 'negative_payment_value': 0, 'negative_payment_installments': 0}
Payments Silver persistence and quality validation passed.


In [23]:
from pyspark.sql import functions as F

bronze_path = BRONZE_DIR / "olist_order_reviews_dataset"
silver_path = SILVER_DIR / "reviews"

# Protect existing data.
if silver_path.exists():
    if any(silver_path.iterdir()):
        raise RuntimeError(
            f"{silver_path} is not empty. Refusing to overwrite existing data."
        )
    silver_path.rmdir()

# Read Bronze.
reviews_silver = spark.read.parquet(str(bronze_path))

# Standardize column names.
for column in reviews_silver.columns:
    standardized_name = column.strip().lower()
    if column != standardized_name:
        reviews_silver = reviews_silver.withColumnRenamed(
            column, standardized_name
        )

# Trim string values while preserving nulls.
for column, data_type in reviews_silver.dtypes:
    if data_type == "string":
        reviews_silver = reviews_silver.withColumn(
            column, F.trim(F.col(column))
        )

# Persist Silver.
reviews_silver.write.mode("errorifexists").parquet(str(silver_path))

# Reload and validate.
bronze_count = spark.read.parquet(str(bronze_path)).count()
reviews_reloaded = spark.read.parquet(str(silver_path))
silver_count = reviews_reloaded.count()

quality = reviews_reloaded.agg(
    F.sum(F.when(F.col("review_id").isNull(), 1).otherwise(0))
        .alias("null_review_id"),
    F.sum(F.when(F.col("order_id").isNull(), 1).otherwise(0))
        .alias("null_order_id"),
    F.sum(
        F.when(
            F.col("review_score").isNotNull()
            & (~F.col("review_score").between(1, 5)),
            1
        ).otherwise(0)
    ).alias("invalid_review_score"),
).first().asDict()

print(f"Bronze rows: {bronze_count}")
print(f"Silver rows: {silver_count}")
print(f"Row counts match: {bronze_count == silver_count}")
print("Silver columns:", reviews_reloaded.columns)
reviews_reloaded.printSchema()
print("Quality checks:", quality)

assert bronze_count == silver_count, "Review row count mismatch"

print(
    "Review records preserved. Review-key issues are reported for "
    "documented handling in downstream transformations."
)

{"ts": "2026-10-17 05:11:45.705", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '2018-04-01 00:27:51' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 48 in cell [23]", "line": "", "fragment": "__ge__", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o1004.collectToPython.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value '2018-04-01 00:27:51' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"__ge__\" was called from\nline 48 in cell [23]\n\n\tat 

NumberFormatException: [CAST_INVALID_INPUT] The value '2018-04-01 00:27:51' of the type "STRING" cannot be cast to "BIGINT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"__ge__" was called from
line 48 in cell [23]


In [25]:
from pathlib import Path

raw_path = RAW_DIR / "olist_order_reviews_dataset.csv"
bronze_path = BRONZE_DIR / "olist_order_reviews_dataset"

print("Raw CSV header and sample records:")
with open(raw_path, "r", encoding="utf-8") as file:
    for _ in range(6):
        line = file.readline()
        if not line:
            break
        print(repr(line.rstrip("\n")))

print("\nBronze review sample:")
bronze_reviews = spark.read.parquet(str(bronze_path))
bronze_reviews.select(
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp",
).show(10, truncate=False)

Raw CSV header and sample records:
'"review_id","order_id","review_score","review_comment_title","review_comment_message","review_creation_date","review_answer_timestamp"'
'"7bc2406110b926393aa56f80a40eba40","73fc7af87114b39712e6da79b0a377eb",4,,,2018-01-18 00:00:00,2018-01-18 21:46:59'
'"80e641a11e56f04c1ad469d5645fdfde","a548910a1c6147796b98fdf73dbeba33",5,,,2018-03-10 00:00:00,2018-03-11 03:05:13'
'"228ce5500dc1d8e020d8d1322874b6f0","f9e4b658b201a9f2ecdecbb34bed034b",5,,,2018-02-17 00:00:00,2018-02-18 14:36:24'
'"e64fb393e7b32834bb789ff8bb30750e","658677c97b385a9be170737859d3511b",5,,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06'
'"f7c4243c7fe1938f181bec41a392bdeb","8e6bfb81e283fa7e4f11123a3fb894f1",5,,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01 00:00:00,2018-03-02 10:26:53'

Bronze review sample:
+--------------------------------+--------------------------------+------------+--------

In [33]:
from pyspark.sql import functions as F

bronze_path = BRONZE_DIR / "olist_products_dataset"
silver_path = SILVER_DIR / "products"

# Protect existing data.
if silver_path.exists():
    if any(silver_path.iterdir()):
        raise RuntimeError(
            f"{silver_path} is not empty. Refusing to overwrite existing data."
        )
    silver_path.rmdir()

# Read Bronze.
products_silver = spark.read.parquet(str(bronze_path))

# Standardize column names.
for column in products_silver.columns:
    standardized_name = column.strip().lower()
    if column != standardized_name:
        products_silver = products_silver.withColumnRenamed(
            column, standardized_name
        )

# Trim string values while preserving nulls.
for column, data_type in products_silver.dtypes:
    if data_type == "string":
        products_silver = products_silver.withColumn(
            column, F.trim(F.col(column))
        )

# Persist Silver.
products_silver.write.mode("errorifexists").parquet(str(silver_path))

# Reload and validate.
bronze_count = spark.read.parquet(str(bronze_path)).count()
products_reloaded = spark.read.parquet(str(silver_path))
silver_count = products_reloaded.count()

quality = products_reloaded.agg(
    F.sum(F.when(F.col("product_id").isNull(), 1).otherwise(0))
        .alias("null_product_id"),
    F.sum(
        F.when(
            F.col("product_weight_g").isNotNull()
            & (F.col("product_weight_g") < 0),
            1,
        ).otherwise(0)
    ).alias("negative_product_weight_g"),
).first().asDict()

print(f"Bronze rows: {bronze_count}")
print(f"Silver rows: {silver_count}")
print(f"Row counts match: {bronze_count == silver_count}")
print("Silver columns:", products_reloaded.columns)
products_reloaded.printSchema()
print("Quality checks:", quality)

assert bronze_count == silver_count, "Product row count mismatch"
assert quality["null_product_id"] == 0, "Null product IDs detected"
assert quality["negative_product_weight_g"] == 0, "Negative product weights detected"

print("Products Silver persistence and quality validation passed.")

Bronze rows: 32951
Silver rows: 32951
Row counts match: True
Silver columns: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)

Quality checks: {'null_product_id': 0, 'negative_product_weight_g': 0}
Products Silver persistence and quality validation passed.


In [34]:
from pyspark.sql import functions as F

bronze_path = BRONZE_DIR / "olist_sellers_dataset"
silver_path = SILVER_DIR / "sellers"

# Protect existing data.
if silver_path.exists():
    if any(silver_path.iterdir()):
        raise RuntimeError(
            f"{silver_path} is not empty. Refusing to overwrite existing data."
        )
    silver_path.rmdir()

# Read Bronze.
sellers_silver = spark.read.parquet(str(bronze_path))

# Standardize column names.
for column in sellers_silver.columns:
    standardized_name = column.strip().lower()
    if column != standardized_name:
        sellers_silver = sellers_silver.withColumnRenamed(
            column, standardized_name
        )

# Trim string values while preserving nulls.
for column, data_type in sellers_silver.dtypes:
    if data_type == "string":
        sellers_silver = sellers_silver.withColumn(
            column, F.trim(F.col(column))
        )

# Persist Silver.
sellers_silver.write.mode("errorifexists").parquet(str(silver_path))

# Reload and validate.
bronze_count = spark.read.parquet(str(bronze_path)).count()
sellers_reloaded = spark.read.parquet(str(silver_path))
silver_count = sellers_reloaded.count()

quality = sellers_reloaded.agg(
    F.sum(F.when(F.col("seller_id").isNull(), 1).otherwise(0))
        .alias("null_seller_id"),
).first().asDict()

print(f"Bronze rows: {bronze_count}")
print(f"Silver rows: {silver_count}")
print(f"Row counts match: {bronze_count == silver_count}")
print("Silver columns:", sellers_reloaded.columns)
sellers_reloaded.printSchema()
print("Quality checks:", quality)

assert bronze_count == silver_count, "Seller row count mismatch"
assert quality["null_seller_id"] == 0, "Null seller IDs detected"

print("Sellers Silver persistence and quality validation passed.")

Bronze rows: 3095
Silver rows: 3095
Row counts match: True
Silver columns: ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']
root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)

Quality checks: {'null_seller_id': 0}
Sellers Silver persistence and quality validation passed.


In [35]:
from pyspark.sql import functions as F

bronze_path = BRONZE_DIR / "olist_geolocation_dataset"
silver_path = SILVER_DIR / "geolocation"

# Protect existing data.
if silver_path.exists():
    if any(silver_path.iterdir()):
        raise RuntimeError(
            f"{silver_path} is not empty. Refusing to overwrite existing data."
        )
    silver_path.rmdir()

# Read Bronze.
geo_silver = spark.read.parquet(str(bronze_path))

# Standardize column names.
for column in geo_silver.columns:
    standardized_name = column.strip().lower()
    if column != standardized_name:
        geo_silver = geo_silver.withColumnRenamed(
            column, standardized_name
        )

# Trim string values while preserving nulls.
for column, data_type in geo_silver.dtypes:
    if data_type == "string":
        geo_silver = geo_silver.withColumn(
            column, F.trim(F.col(column))
        )

# Persist Silver.
geo_silver.write.mode("errorifexists").parquet(str(silver_path))

# Reload and validate.
bronze_count = spark.read.parquet(str(bronze_path)).count()
geo_reloaded = spark.read.parquet(str(silver_path))
silver_count = geo_reloaded.count()

quality = geo_reloaded.agg(
    F.sum(
        F.when(F.col("geolocation_zip_code_prefix").isNull(), 1)
        .otherwise(0)
    ).alias("null_zip_prefix"),
    F.sum(
        F.when(
            F.col("geolocation_lat").isNotNull()
            & (~F.col("geolocation_lat").between(-90, 90)),
            1,
        ).otherwise(0)
    ).alias("invalid_latitude"),
    F.sum(
        F.when(
            F.col("geolocation_lng").isNotNull()
            & (~F.col("geolocation_lng").between(-180, 180)),
            1,
        ).otherwise(0)
    ).alias("invalid_longitude"),
).first().asDict()

print(f"Bronze rows: {bronze_count}")
print(f"Silver rows: {silver_count}")
print(f"Row counts match: {bronze_count == silver_count}")
print("Silver columns:", geo_reloaded.columns)
geo_reloaded.printSchema()
print("Quality checks:", quality)

assert bronze_count == silver_count, "Geolocation row count mismatch"

print("Geolocation Silver persistence and row-count validation passed.")

Bronze rows: 1000163
Silver rows: 1000163
Row counts match: True
Silver columns: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']
root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)

Quality checks: {'null_zip_prefix': 0, 'invalid_latitude': 0, 'invalid_longitude': 0}
Geolocation Silver persistence and row-count validation passed.


In [36]:
from pyspark.sql import functions as F

bronze_path = BRONZE_DIR / "product_category_name_translation"
silver_path = SILVER_DIR / "category_translation"

# Protect existing data.
if silver_path.exists():
    if any(silver_path.iterdir()):
        raise RuntimeError(
            f"{silver_path} is not empty. Refusing to overwrite existing data."
        )
    silver_path.rmdir()

# Read Bronze.
translation_silver = spark.read.parquet(str(bronze_path))

# Standardize column names.
for column in translation_silver.columns:
    standardized_name = column.strip().lower()
    if column != standardized_name:
        translation_silver = translation_silver.withColumnRenamed(
            column, standardized_name
        )

# Trim string values while preserving nulls.
for column, data_type in translation_silver.dtypes:
    if data_type == "string":
        translation_silver = translation_silver.withColumn(
            column, F.trim(F.col(column))
        )

# Persist Silver.
translation_silver.write.mode("errorifexists").parquet(str(silver_path))

# Reload and validate.
bronze_count = spark.read.parquet(str(bronze_path)).count()
translation_reloaded = spark.read.parquet(str(silver_path))
silver_count = translation_reloaded.count()

quality = translation_reloaded.agg(
    F.sum(
        F.when(F.col("product_category_name").isNull(), 1).otherwise(0)
    ).alias("null_source_category"),
    F.sum(
        F.when(
            F.col("product_category_name_english").isNull(), 1
        ).otherwise(0)
    ).alias("null_english_category"),
).first().asDict()

duplicate_source_categories = (
    translation_reloaded
    .groupBy("product_category_name")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Bronze rows: {bronze_count}")
print(f"Silver rows: {silver_count}")
print(f"Row counts match: {bronze_count == silver_count}")
print("Silver columns:", translation_reloaded.columns)
translation_reloaded.printSchema()
print("Quality checks:", quality)
print(f"Duplicate source categories: {duplicate_source_categories}")

assert bronze_count == silver_count, "Translation row count mismatch"
assert quality["null_source_category"] == 0, "Null source categories detected"
assert duplicate_source_categories == 0, "Duplicate source categories detected"

print("Category translation Silver persistence and validation passed.")

Bronze rows: 71
Silver rows: 71
Row counts match: True
Silver columns: ['product_category_name', 'product_category_name_english']
root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)

Quality checks: {'null_source_category': 0, 'null_english_category': 0}
Duplicate source categories: 0
Category translation Silver persistence and validation passed.


Varify all nine silver dataset

In [37]:
silver_mapping = {
    "olist_customers_dataset": "customers",
    "olist_geolocation_dataset": "geolocation",
    "olist_order_items_dataset": "order_items",
    "olist_order_payments_dataset": "payments",
    "olist_order_reviews_dataset": "reviews",
    "olist_orders_dataset": "orders",
    "olist_products_dataset": "products",
    "olist_sellers_dataset": "sellers",
    "product_category_name_translation": "category_translation",
}

silver_results = []

for bronze_name, silver_name in silver_mapping.items():
    bronze_path = BRONZE_DIR / bronze_name
    silver_path = SILVER_DIR / silver_name

    bronze_count = spark.read.parquet(str(bronze_path)).count()

    if silver_path.exists() and any(silver_path.iterdir()):
        silver_df = spark.read.parquet(str(silver_path))
        silver_count = silver_df.count()
        readable = True
    else:
        silver_count = None
        readable = False

    count_match = readable and bronze_count == silver_count

    silver_results.append({
        "dataset": silver_name,
        "bronze_rows": bronze_count,
        "silver_rows": silver_count,
        "readable": readable,
        "row_count_match": count_match,
    })

for result in silver_results:
    status = "PASS" if result["row_count_match"] else "REVIEW"
    print(
        f"{result['dataset']}: {status} | "
        f"Bronze={result['bronze_rows']} | "
        f"Silver={result['silver_rows']} | "
        f"Readable={result['readable']}"
    )

passed = sum(r["row_count_match"] for r in silver_results)
print(f"\nRow-count matches: {passed}/{len(silver_results)}")

review_result = next(
    r for r in silver_results if r["dataset"] == "reviews"
)
print(
    "\nReview discrepancy:",
    review_result["bronze_rows"] - review_result["silver_rows"],
    "rows"
)

customers: PASS | Bronze=99441 | Silver=99441 | Readable=True
geolocation: PASS | Bronze=1000163 | Silver=1000163 | Readable=True
order_items: PASS | Bronze=112650 | Silver=112650 | Readable=True
payments: PASS | Bronze=103886 | Silver=103886 | Readable=True
reviews: PASS | Bronze=104162 | Silver=104162 | Readable=True
orders: PASS | Bronze=99441 | Silver=99441 | Readable=True
products: PASS | Bronze=32951 | Silver=32951 | Readable=True
sellers: PASS | Bronze=3095 | Silver=3095 | Readable=True
category_translation: PASS | Bronze=71 | Silver=71 | Readable=True

Row-count matches: 9/9

Review discrepancy: 0 rows


separate recovered review dataset

In [38]:
import csv
import io
import re
from pyspark.sql import functions as F

raw_path = RAW_DIR / "olist_order_reviews_dataset.csv"
recovered_path = SILVER_DIR / "reviews_recovered"

record_start = re.compile(
    r'^"[0-9a-fA-F]{32}","[0-9a-fA-F]{32}",'
)
timestamp_pattern = re.compile(
    r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"
)

# Protect any existing recovery output.
if recovered_path.exists():
    if any(recovered_path.iterdir()):
        raise RuntimeError(
            f"{recovered_path} already contains data. "
            "Refusing to overwrite it."
        )
    recovered_path.rmdir()

# Read physical lines and reconstruct records using identifier anchors.
with open(raw_path, "r", encoding="utf-8", newline="") as file:
    lines = file.readlines()

header = next(csv.reader(io.StringIO(lines[0]), quotechar='"'))
record_texts = []
current_lines = []

for line in lines[1:]:
    if record_start.match(line):
        if current_lines:
            record_texts.append("".join(current_lines))
        current_lines = [line]
    elif current_lines:
        current_lines.append(line)

if current_lines:
    record_texts.append("".join(current_lines))

# Parse and validate every reconstructed record before writing.
records = []

for index, record_text in enumerate(record_texts, start=1):
    row = next(csv.reader(
        io.StringIO(record_text),
        delimiter=",",
        quotechar='"',
        doublequote=True,
        strict=False,
    ))

    if len(row) != 7:
        raise ValueError(f"Record {index} has {len(row)} fields.")

    if row[2].strip() not in {"1", "2", "3", "4", "5"}:
        raise ValueError(f"Record {index} has an invalid review score.")

    if not timestamp_pattern.fullmatch(row[5].strip()):
        raise ValueError(f"Record {index} has an invalid creation timestamp.")

    if not timestamp_pattern.fullmatch(row[6].strip()):
        raise ValueError(f"Record {index} has an invalid answer timestamp.")

    records.append(row)

print(f"Reconstructed records validated: {len(records)}")

# Build a DataFrame without losing multiline comment text.
recovered_df = spark.createDataFrame(records, schema=header)

recovered_df = (
    recovered_df
    .withColumn("review_score", F.col("review_score").cast("int"))
    .withColumn(
        "review_creation_date",
        F.try_to_timestamp(
            F.col("review_creation_date"),
            F.lit("yyyy-MM-dd HH:mm:ss"),
        ),
    )
    .withColumn(
        "review_answer_timestamp",
        F.try_to_timestamp(
            F.col("review_answer_timestamp"),
            F.lit("yyyy-MM-dd HH:mm:ss"),
        ),
    )
)

# Persist to a separate path, then reload for validation.
recovered_df.write.mode("errorifexists").parquet(str(recovered_path))
reloaded = spark.read.parquet(str(recovered_path))

print(f"Recovered Parquet rows: {reloaded.count()}")
print("Recovered schema:")
reloaded.printSchema()

reloaded.select(
    F.count("*").alias("rows"),
    F.sum(F.when(F.col("review_id").isNull(), 1).otherwise(0))
        .alias("null_review_id"),
    F.sum(F.when(F.col("order_id").isNull(), 1).otherwise(0))
        .alias("null_order_id"),
    F.sum(
        F.when(~F.col("review_score").between(1, 5), 1).otherwise(0)
    ).alias("invalid_scores"),
    F.sum(F.when(F.col("review_creation_date").isNull(), 1).otherwise(0))
        .alias("null_creation_timestamp"),
    F.sum(F.when(F.col("review_answer_timestamp").isNull(), 1).otherwise(0))
        .alias("null_answer_timestamp"),
).show(truncate=False)

print("Existing Bronze and Silver review outputs were not modified.")

Reconstructed records validated: 99224
Recovered Parquet rows: 99224
Recovered schema:
root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)

+-----+--------------+-------------+--------------+-----------------------+---------------------+
|rows |null_review_id|null_order_id|invalid_scores|null_creation_timestamp|null_answer_timestamp|
+-----+--------------+-------------+--------------+-----------------------+---------------------+
|99224|0             |0            |0             |0                      |0                    |
+-----+--------------+-------------+--------------+-----------------------+---------------------+

Existing Bronze and Silver review outputs were not modified.


In [45]:
from pathlib import Path
import shutil

RUNTIME_SILVER = Path("/content/data/silver")
REPO_SILVER = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline/data/silver"
)

silver_datasets = [
    "customers",
    "geolocation",
    "order_items",
    "payments",
    "reviews",
    "reviews_recovered",
    "orders",
    "products",
    "sellers",
    "category_translation",
]

for dataset in silver_datasets:
    source = RUNTIME_SILVER / dataset
    destination = REPO_SILVER / dataset

    source_files = [
        p for p in source.rglob("*.parquet")
        if p.is_file()
    ] if source.is_dir() else []

    if not source_files:
        print(f"{dataset}: SKIPPED — no runtime Parquet files")
        continue

    if destination.exists() and any(destination.iterdir()):
        print(f"{dataset}: SKIPPED — destination already contains data")
        continue

    destination.mkdir(parents=True, exist_ok=True)

    for source_file in source_files:
        relative_path = source_file.relative_to(source)
        target_file = destination / relative_path
        target_file.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_file, target_file)

    copied_files = [
        p for p in destination.rglob("*.parquet")
        if p.is_file()
    ]

    print(
        f"{dataset}: copied {len(source_files)} Parquet files; "
        f"destination has {len(copied_files)}"
    )

customers: copied 2 Parquet files; destination has 2
geolocation: copied 2 Parquet files; destination has 2
order_items: copied 2 Parquet files; destination has 2
payments: copied 2 Parquet files; destination has 2
reviews: copied 2 Parquet files; destination has 2
reviews_recovered: copied 2 Parquet files; destination has 2
orders: copied 2 Parquet files; destination has 2
products: copied 1 Parquet files; destination has 1
sellers: copied 1 Parquet files; destination has 1
category_translation: copied 1 Parquet files; destination has 1


Build the **Gold Analytics Layer**

In [47]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
GOLD_ANALYTICS_DIR = DATA_DIR / "gold" / "analytics"

analytics_files = [
    "order_analytics.csv",
    "product_analytics.csv",
    "category_analytics.csv",
    "customer_analytics.csv",
    "monthly_sales_analytics.csv",
    "order_status_summary.csv",
]

print("Project directory:", PROJECT_DIR)
print("Project directory exists:", PROJECT_DIR.is_dir())
print("Processed directory exists:", PROCESSED_DIR.is_dir())
print("Gold analytics directory:", GOLD_ANALYTICS_DIR)
print("Gold analytics directory exists:", GOLD_ANALYTICS_DIR.is_dir())

print("\nSource CSV files:")
for filename in analytics_files:
    path = PROCESSED_DIR / filename
    print(f"{filename}: {'FOUND' if path.is_file() else 'MISSING'}")

print("\nExisting Gold analytics outputs:")
if GOLD_ANALYTICS_DIR.is_dir():
    entries = list(GOLD_ANALYTICS_DIR.iterdir())

    if not entries:
        print("Directory is empty.")
    else:
        for entry in entries:
            parquet_count = (
                sum(1 for p in entry.rglob("*.parquet") if p.is_file())
                if entry.is_dir()
                else 0
            )
            print(f"{entry.name}: Parquet files={parquet_count}")
else:
    print("Directory does not exist yet.")

Project directory: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline
Project directory exists: True
Processed directory exists: True
Gold analytics directory: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/data/gold/analytics
Gold analytics directory exists: True

Source CSV files:
order_analytics.csv: FOUND
product_analytics.csv: FOUND
category_analytics.csv: FOUND
customer_analytics.csv: FOUND
monthly_sales_analytics.csv: FOUND
order_status_summary.csv: FOUND

Existing Gold analytics outputs:
Directory is empty.


Convert analytics CSVs to Parquet

In [48]:
from pathlib import Path
from pyspark.sql import SparkSession

# Project paths
PROJECT_DIR = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
GOLD_ANALYTICS_DIR = DATA_DIR / "gold" / "analytics"

GOLD_ANALYTICS_DIR.mkdir(parents=True, exist_ok=True)

# Reuse the existing Spark session if available
spark = (
    SparkSession.builder
    .appName("SalesAnalyticsGoldLayer")
    .getOrCreate()
)

analytics_files = [
    "order_analytics.csv",
    "product_analytics.csv",
    "category_analytics.csv",
    "customer_analytics.csv",
    "monthly_sales_analytics.csv",
    "order_status_summary.csv",
]

results = []
errors = []

for filename in analytics_files:
    source_path = PROCESSED_DIR / filename
    dataset_name = source_path.stem
    output_path = GOLD_ANALYTICS_DIR / dataset_name

    try:
        if not source_path.is_file():
            raise FileNotFoundError(f"Source CSV not found: {source_path}")

        if output_path.exists() and any(output_path.iterdir()):
            raise FileExistsError(
                f"Output already contains files; refusing to overwrite: "
                f"{output_path}"
            )

        df = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(str(source_path))
        )

        source_rows = df.count()

        (
            df.write
            .mode("errorifexists")
            .parquet(str(output_path))
        )

        saved_df = spark.read.parquet(str(output_path))
        saved_rows = saved_df.count()

        passed = source_rows == saved_rows

        results.append({
            "dataset": dataset_name,
            "source_rows": source_rows,
            "parquet_rows": saved_rows,
            "columns": len(saved_df.columns),
            "status": "PASS" if passed else "FAIL",
        })

        if not passed:
            errors.append(f"{dataset_name}: row-count mismatch")

    except Exception as exc:
        errors.append(f"{dataset_name}: {type(exc).__name__}: {exc}")

print("GOLD ANALYTICS — CSV TO PARQUET")
print("-" * 65)

for result in results:
    print(
        f"{result['dataset']}: "
        f"source_rows={result['source_rows']}, "
        f"parquet_rows={result['parquet_rows']}, "
        f"columns={result['columns']}, "
        f"status={result['status']}"
    )

print("\nErrors:", errors if errors else "None")
print("Datasets passed:", sum(r["status"] == "PASS" for r in results))
print("Datasets expected:", len(analytics_files))

GOLD ANALYTICS — CSV TO PARQUET
-----------------------------------------------------------------
order_analytics: source_rows=99441, parquet_rows=99441, columns=21, status=PASS
product_analytics: source_rows=32729, parquet_rows=32729, columns=7, status=PASS
category_analytics: source_rows=72, parquet_rows=72, columns=7, status=PASS
customer_analytics: source_rows=94990, parquet_rows=94990, columns=9, status=PASS
monthly_sales_analytics: source_rows=24, parquet_rows=24, columns=7, status=PASS
order_status_summary: source_rows=8, parquet_rows=8, columns=2, status=PASS

Errors: None
Datasets passed: 6
Datasets expected: 6


Varify Silver sources datasets

In [49]:
from pathlib import Path
from pyspark.sql import SparkSession

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

SILVER_DIR = PROJECT_DIR / "data" / "silver"

spark = (
    SparkSession.builder
    .appName("SalesAnalyticsGoldDimensionalModel")
    .getOrCreate()
)

expected_datasets = {
    "customers": 99441,
    "orders": 99441,
    "order_items": 112650,
    "products": 32951,
    "sellers": 3095,
}

print("Silver source verification")
print("-" * 60)

all_passed = True

for dataset, expected_rows in expected_datasets.items():
    dataset_path = SILVER_DIR / dataset

    try:
        if not dataset_path.is_dir():
            raise FileNotFoundError(f"Missing directory: {dataset_path}")

        df = spark.read.parquet(str(dataset_path))
        actual_rows = df.count()
        passed = actual_rows == expected_rows
        all_passed = all_passed and passed

        print(
            f"{dataset}: rows={actual_rows}, "
            f"expected={expected_rows}, "
            f"columns={len(df.columns)}, "
            f"status={'PASS' if passed else 'FAIL'}"
        )

    except Exception as exc:
        all_passed = False
        print(f"{dataset}: ERROR - {type(exc).__name__}: {exc}")

print("-" * 60)
print("Overall:", "PASS" if all_passed else "CHECK REQUIRED")

Silver source verification
------------------------------------------------------------
customers: rows=99441, expected=99441, columns=5, status=PASS
orders: rows=99441, expected=99441, columns=8, status=PASS
order_items: rows=112650, expected=112650, columns=7, status=PASS
products: rows=32951, expected=32951, columns=9, status=PASS
sellers: rows=3095, expected=3095, columns=4, status=PASS
------------------------------------------------------------
Overall: PASS


### Build Dimensional Model

In [54]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

SILVER_DIR = PROJECT_DIR / "data" / "silver"
GOLD_DIMENSIONAL_DIR = PROJECT_DIR / "data" / "gold" / "dimensional"

spark = (
    SparkSession.builder
    .appName("SalesAnalyticsGoldDimensionalModel")
    .getOrCreate()
)

dimension_specs = {
    "dim_customer": {
        "source": "customers",
        "business_key": "customer_id",
        "attributes": [
            "customer_unique_id",
            "customer_zip_code_prefix",
            "customer_city",
            "customer_state",
        ],
    },
    "dim_product": {
        "source": "products",
        "business_key": "product_id",
        "attributes": [
            "product_category_name",
            "product_name_lenght",
            "product_description_lenght",
            "product_photos_qty",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm",
        ],
    },
    "dim_seller": {
        "source": "sellers",
        "business_key": "seller_id",
        "attributes": [
            "seller_zip_code_prefix",
            "seller_city",
            "seller_state",
        ],
    },
    "dim_order": {
        "source": "orders",
        "business_key": "order_id",
        "attributes": [
            "customer_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
        ],
    },
}

GOLD_DIMENSIONAL_DIR.mkdir(parents=True, exist_ok=True)

# Safety check: do not overwrite existing dimension data.
for name in dimension_specs:
    output_path = GOLD_DIMENSIONAL_DIR / name

    if output_path.exists():
        if not output_path.is_dir():
            raise RuntimeError(f"Output path is not a directory: {output_path}")

        existing_files = [
            p for p in output_path.rglob("*") if p.is_file()
        ]

        if existing_files:
            raise RuntimeError(
                f"Existing files found in {output_path}; "
                "stopping to protect existing data."
            )

results = []
errors = []

for name, spec in dimension_specs.items():
    output_path = GOLD_DIMENSIONAL_DIR / name
    business_key = spec["business_key"]

    try:
        source_df = spark.read.parquet(
            str(SILVER_DIR / spec["source"])
        )

        required = [business_key] + spec["attributes"]
        missing = [c for c in required if c not in source_df.columns]

        if missing:
            raise ValueError(f"Missing columns: {missing}")

        source_rows = source_df.count()

        null_keys = source_df.filter(
            F.col(business_key).isNull()
        ).count()

        duplicate_groups = (
            source_df.groupBy(business_key)
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

        if null_keys or duplicate_groups:
            raise ValueError(
                f"Invalid keys: nulls={null_keys}, "
                f"duplicate_groups={duplicate_groups}"
            )

        key_window = Window.orderBy(F.col(business_key))

        dimension_df = (
            source_df.select(*required)
            .withColumn(
                f"{name}_key",
                F.row_number().over(key_window).cast("long"),
            )
            .select(
                f"{name}_key",
                business_key,
                *spec["attributes"],
            )
        )

        # Safe because the preflight check confirmed these directories
        # contain no files.
        dimension_df.write.mode("overwrite").parquet(str(output_path))

        saved_df = spark.read.parquet(str(output_path))
        saved_rows = saved_df.count()

        null_keys_after = saved_df.filter(
            F.col(business_key).isNull()
        ).count()

        duplicate_groups_after = (
            saved_df.groupBy(business_key)
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

        status = (
            "PASS"
            if source_rows == saved_rows
            and null_keys_after == 0
            and duplicate_groups_after == 0
            else "FAIL"
        )

        results.append(
            (name, source_rows, saved_rows, len(saved_df.columns), status)
        )

        if status != "PASS":
            errors.append(f"{name}: output validation failed")

    except Exception as exc:
        errors.append(f"{name}: {type(exc).__name__}: {exc}")

print("GOLD DIMENSION VALIDATION")
print("-" * 75)

for name, source_rows, saved_rows, columns, status in results:
    print(
        f"{name}: source_rows={source_rows}, "
        f"saved_rows={saved_rows}, columns={columns}, status={status}"
    )

print("\nErrors:", errors if errors else "None")
print("Dimensions passed:", sum(r[4] == "PASS" for r in results))
print("Dimensions expected:", len(dimension_specs))

GOLD DIMENSION VALIDATION
---------------------------------------------------------------------------
dim_customer: source_rows=99441, saved_rows=99441, columns=6, status=PASS
dim_product: source_rows=32951, saved_rows=32951, columns=10, status=PASS
dim_seller: source_rows=3095, saved_rows=3095, columns=5, status=PASS
dim_order: source_rows=99441, saved_rows=99441, columns=9, status=PASS

Errors: None
Dimensions passed: 4
Dimensions expected: 4


## Build fact_sales





In [56]:
#check fact-table output path
from pathlib import Path
from pyspark.sql import SparkSession

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

DATA_DIR = PROJECT_DIR / "data"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIMENSIONAL_DIR = DATA_DIR / "gold" / "dimensional"
FACT_PATH = GOLD_DIMENSIONAL_DIR / "fact_sales"

spark = (
    SparkSession.builder
    .appName("SalesAnalyticsFactSales")
    .getOrCreate()
)

print("Fact output path:", FACT_PATH)
print("Path exists:", FACT_PATH.exists())

if FACT_PATH.exists():
    existing_files = [
        p for p in FACT_PATH.rglob("*") if p.is_file()
    ]
    print("Existing files:", len(existing_files))

    if existing_files:
        print("Existing fact data will be preserved.")

print("\nOrder items schema:")
order_items = spark.read.parquet(str(SILVER_DIR / "order_items"))
order_items.printSchema()

print("\nRequired dimension keys:")
for name in [
    "dim_order",
    "dim_customer",
    "dim_product",
    "dim_seller",
]:
    path = GOLD_DIMENSIONAL_DIR / name
    df = spark.read.parquet(str(path))
    print(f"{name}: rows={df.count()}, columns={df.columns}")

Fact output path: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/data/gold/dimensional/fact_sales
Path exists: True
Existing files: 0

Order items schema:
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)


Required dimension keys:
dim_order: rows=99441, columns=['dim_order_key', 'order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
dim_customer: rows=99441, columns=['dim_customer_key', 'customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
dim_product: rows=32951, columns=['dim_product_key', 'product_id', 'product_ca

Create the Gold sales fact tables

In [57]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

DATA_DIR = PROJECT_DIR / "data"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIMENSIONAL_DIR = DATA_DIR / "gold" / "dimensional"
FACT_PATH = GOLD_DIMENSIONAL_DIR / "fact_sales"

spark = (
    SparkSession.builder
    .appName("SalesAnalyticsFactSales")
    .getOrCreate()
)

# Protect any fact data that may have appeared since the previous inspection.
if FACT_PATH.exists():
    existing_files = [
        p for p in FACT_PATH.rglob("*") if p.is_file()
    ]
    if existing_files:
        raise RuntimeError(
            f"Fact output is no longer empty; refusing to overwrite: {FACT_PATH}"
        )

# Read Silver source
items = spark.read.parquet(str(SILVER_DIR / "order_items"))

# Read validated Gold dimensions
dim_order = spark.read.parquet(
    str(GOLD_DIMENSIONAL_DIR / "dim_order")
).select(
    "dim_order_key", "order_id", "customer_id"
)

dim_customer = spark.read.parquet(
    str(GOLD_DIMENSIONAL_DIR / "dim_customer")
).select(
    "dim_customer_key", "customer_id"
)

dim_product = spark.read.parquet(
    str(GOLD_DIMENSIONAL_DIR / "dim_product")
).select(
    "dim_product_key", "product_id"
)

dim_seller = spark.read.parquet(
    str(GOLD_DIMENSIONAL_DIR / "dim_seller")
).select(
    "dim_seller_key", "seller_id"
)

# Source key checks
source_rows = items.count()

null_source_keys = items.filter(
    F.col("order_id").isNull() |
    F.col("order_item_id").isNull()
).count()

duplicate_source_keys = (
    items.groupBy("order_id", "order_item_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if null_source_keys or duplicate_source_keys:
    raise ValueError(
        f"Invalid order-item keys: nulls={null_source_keys}, "
        f"duplicate_groups={duplicate_source_keys}"
    )

# Measure baselines. Keep null measures null; do not silently treat them as zero.
source_metrics = items.agg(
    F.sum("price").alias("price_total"),
    F.sum("freight_value").alias("freight_total"),
    F.count("price").alias("price_non_null_count"),
    F.count("freight_value").alias("freight_non_null_count"),
).first().asDict()

# Join dimensions. Left joins preserve source items so unmatched references
# can be detected instead of silently dropping records.
joined = (
    items
    .join(dim_order, on="order_id", how="left")
    .join(dim_customer, on="customer_id", how="left")
    .join(dim_product, on="product_id", how="left")
    .join(dim_seller, on="seller_id", how="left")
)

# Check that every fact row resolves to all required dimensions.
unmatched = joined.filter(
    F.col("dim_order_key").isNull() |
    F.col("dim_customer_key").isNull() |
    F.col("dim_product_key").isNull() |
    F.col("dim_seller_key").isNull()
).count()

if unmatched:
    raise ValueError(
        f"Found {unmatched} order items with unmatched dimension references. "
        "Fact output was not written."
    )

# Create one fact row per source order item.
fact_window = Window.orderBy("order_id", "order_item_id")

fact = (
    joined
    .withColumn(
        "fact_sales_key",
        F.row_number().over(fact_window).cast("long")
    )
    .select(
        "fact_sales_key",
        "order_id",
        "order_item_id",
        "dim_order_key",
        "dim_customer_key",
        "dim_product_key",
        "dim_seller_key",
        "shipping_limit_date",
        "price",
        "freight_value",
        F.lit(1).cast("integer").alias("quantity"),
    )
)

# Validate in memory before writing.
fact_rows = fact.count()

duplicate_fact_keys = (
    fact.groupBy("order_id", "order_item_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

fact_metrics = fact.agg(
    F.sum("price").alias("price_total"),
    F.sum("freight_value").alias("freight_total"),
    F.count("price").alias("price_non_null_count"),
    F.count("freight_value").alias("freight_non_null_count"),
).first().asDict()

row_count_ok = fact_rows == source_rows
key_count_ok = duplicate_fact_keys == 0
price_count_ok = (
    fact_metrics["price_non_null_count"]
    == source_metrics["price_non_null_count"]
)
freight_count_ok = (
    fact_metrics["freight_non_null_count"]
    == source_metrics["freight_non_null_count"]
)

def totals_match(source_value, fact_value):
    if source_value is None or fact_value is None:
        return source_value is None and fact_value is None
    return abs(float(source_value) - float(fact_value)) <= 0.01

price_total_ok = totals_match(
    source_metrics["price_total"], fact_metrics["price_total"]
)
freight_total_ok = totals_match(
    source_metrics["freight_total"], fact_metrics["freight_total"]
)

checks = {
    "row_count_matches": row_count_ok,
    "source_item_keys_unique": key_count_ok,
    "all_dimension_references_matched": unmatched == 0,
    "price_non_null_count_matches": price_count_ok,
    "freight_non_null_count_matches": freight_count_ok,
    "price_total_matches": price_total_ok,
    "freight_total_matches": freight_total_ok,
}

print("FACT SALES PRE-WRITE VALIDATION")
print("-" * 65)
print("Source rows:", source_rows)
print("Fact rows:", fact_rows)
print("Unmatched dimension references:", unmatched)
print("Duplicate fact business-key groups:", duplicate_fact_keys)
print("Source price total:", source_metrics["price_total"])
print("Fact price total:", fact_metrics["price_total"])
print("Source freight total:", source_metrics["freight_total"])
print("Fact freight total:", fact_metrics["freight_total"])

for check_name, passed in checks.items():
    print(f"{check_name}: {'PASS' if passed else 'FAIL'}")

if not all(checks.values()):
    raise RuntimeError(
        "Fact pre-write validation failed. No Parquet output was written."
    )

# Recheck destination immediately before writing.
if FACT_PATH.exists() and any(
    p.is_file() for p in FACT_PATH.rglob("*")
):
    raise RuntimeError(
        f"Fact output contains files; refusing to overwrite: {FACT_PATH}"
    )

fact.write.mode("overwrite").parquet(str(FACT_PATH))

# Read-back validation
saved_fact = spark.read.parquet(str(FACT_PATH))
saved_rows = saved_fact.count()

print("\nFACT SALES READ-BACK VALIDATION")
print("-" * 65)
print("Output path:", FACT_PATH)
print("Saved rows:", saved_rows)
print("Saved columns:", saved_fact.columns)
print("Read-back row count:", "PASS" if saved_rows == source_rows else "FAIL")

if saved_rows != source_rows:
    raise RuntimeError("Saved fact row count does not match the source.")

print("\nFact output successfully written and row-count validated.")

FACT SALES PRE-WRITE VALIDATION
-----------------------------------------------------------------
Source rows: 112650
Fact rows: 112650
Unmatched dimension references: 0
Duplicate fact business-key groups: 0
Source price total: 13591643.70000748
Fact price total: 13591643.70000748
Source freight total: 2251909.540000007
Fact freight total: 2251909.540000007
row_count_matches: PASS
source_item_keys_unique: PASS
all_dimension_references_matched: PASS
price_non_null_count_matches: PASS
freight_non_null_count_matches: PASS
price_total_matches: PASS
freight_total_matches: PASS

FACT SALES READ-BACK VALIDATION
-----------------------------------------------------------------
Output path: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/data/gold/dimensional/fact_sales
Saved rows: 112650
Saved columns: ['fact_sales_key', 'order_id', 'order_item_id', 'dim_order_key', 'dim_customer_key', 'dim_product_key', 'dim_seller_key', 'shipping_limit_date', 'price', 'frei

# SQL Validation and analytics recociliation

Silver validation

In [59]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

SILVER_DIR = PROJECT_DIR / "data" / "silver"

spark = (
    SparkSession.builder
    .appName("SalesAnalyticsSilverConsolidatedValidation")
    .getOrCreate()
)

# Expected row counts from our previous Phase 4 checks.
expected_rows = {
    "customers": 99441,
    "orders": 99441,
    "order_items": 112650,
    "payments": 103886,
    "reviews": 104162,
    "geolocation": 1000163,
    "products": 32951,
    "sellers": 3095,
    "category_translation": 71,
}

# Required columns are checked without renaming the source fields.
required_columns = {
    "customers": [
        "customer_id", "customer_unique_id",
        "customer_zip_code_prefix", "customer_city", "customer_state",
    ],
    "orders": [
        "order_id", "customer_id", "order_status",
        "order_purchase_timestamp", "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "order_items": [
        "order_id", "order_item_id", "product_id",
        "seller_id", "shipping_limit_date", "price", "freight_value",
    ],
    "payments": [
        "order_id", "payment_sequential", "payment_type",
        "payment_installments", "payment_value",
    ],
    "reviews": [
        "review_id", "order_id", "review_score",
        "review_comment_title", "review_comment_message",
        "review_creation_date", "review_answer_timestamp",
    ],
    "geolocation": [
        "geolocation_zip_code_prefix", "geolocation_lat",
        "geolocation_lng", "geolocation_city", "geolocation_state",
    ],
    "products": [
        "product_id", "product_category_name",
        "product_name_lenght", "product_description_lenght",
        "product_photos_qty", "product_weight_g",
        "product_length_cm", "product_height_cm", "product_width_cm",
    ],
    "sellers": [
        "seller_id", "seller_zip_code_prefix",
        "seller_city", "seller_state",
    ],
    "category_translation": [
        "product_category_name", "product_category_name_english",
    ],
}

# Load each persisted dataset from the project directory.
datasets = {}
for name in expected_rows:
    path = SILVER_DIR / name
    if not path.is_dir():
        raise FileNotFoundError(f"Silver dataset directory missing: {path}")
    datasets[name] = spark.read.parquet(str(path))

# Collect per-dataset results.
results = []

for name, df in datasets.items():
    actual_rows = df.count()
    missing_columns = sorted(
        set(required_columns[name]) - set(df.columns)
    )

    results.append({
        "dataset": name,
        "expected_rows": expected_rows[name],
        "actual_rows": actual_rows,
        "row_count": "PASS" if actual_rows == expected_rows[name] else "FAIL",
        "missing_columns": missing_columns,
        "schema": "PASS" if not missing_columns else "FAIL",
    })

print("1. ROW COUNT AND SCHEMA VALIDATION")
print("-" * 90)

for r in results:
    print(
        f"{r['dataset']}: rows={r['actual_rows']}/"
        f"{r['expected_rows']}, row_count={r['row_count']}, "
        f"schema={r['schema']}, missing_columns={r['missing_columns']}"
    )

# 2. Primary/composite key checks.
key_specs = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "category_translation": ["product_category_name"],
}

key_results = {}

for name, keys in key_specs.items():
    df = datasets[name]

    null_condition = None
    for key in keys:
        condition = F.col(key).isNull()
        null_condition = (
            condition if null_condition is None
            else null_condition | condition
        )

    null_rows = df.filter(null_condition).count()

    duplicate_groups = (
        df.groupBy(*keys)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    key_results[name] = {
        "null_key_rows": null_rows,
        "duplicate_key_groups": duplicate_groups,
        "status": (
            "PASS" if null_rows == 0 and duplicate_groups == 0
            else "FAIL"
        ),
    }

print("\n2. KEY INTEGRITY")
print("-" * 90)

for name, r in key_results.items():
    print(
        f"{name}: null_key_rows={r['null_key_rows']}, "
        f"duplicate_key_groups={r['duplicate_key_groups']}, "
        f"status={r['status']}"
    )

# 3. Required key/reference fields and numeric constraints.
field_results = {}

for name, columns in {
    "orders": ["order_id", "customer_id"],
    "order_items": ["order_id", "order_item_id", "product_id", "seller_id"],
    "payments": ["order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "category_translation": [
        "product_category_name", "product_category_name_english"
    ],
}.items():
    df = datasets[name]
    null_condition = None

    for column in columns:
        condition = F.col(column).isNull()
        null_condition = (
            condition if null_condition is None
            else null_condition | condition
        )

    field_results[f"{name}_required_fields"] = (
        df.filter(null_condition).count()
    )

numeric_results = {
    "negative_item_price_rows": datasets["order_items"].filter(
        F.col("price") < 0
    ).count(),
    "negative_item_freight_rows": datasets["order_items"].filter(
        F.col("freight_value") < 0
    ).count(),
    "negative_payment_value_rows": datasets["payments"].filter(
        F.col("payment_value") < 0
    ).count(),
    "negative_payment_installment_rows": datasets["payments"].filter(
        F.col("payment_installments") < 0
    ).count(),
    "negative_product_weight_rows": datasets["products"].filter(
        F.col("product_weight_g") < 0
    ).count(),
    "invalid_geolocation_latitude_rows": datasets["geolocation"].filter(
        (F.col("geolocation_lat") < -90) |
        (F.col("geolocation_lat") > 90)
    ).count(),
    "invalid_geolocation_longitude_rows": datasets["geolocation"].filter(
        (F.col("geolocation_lng") < -180) |
        (F.col("geolocation_lng") > 180)
    ).count(),
}

print("\n3. REQUIRED FIELDS")
print("-" * 90)

for name, count in field_results.items():
    print(f"{name}: null_rows={count}, status={'PASS' if count == 0 else 'FAIL'}")

print("\n4. NUMERIC AND RANGE CONSTRAINTS")
print("-" * 90)

for name, count in numeric_results.items():
    print(f"{name}: {count}, status={'PASS' if count == 0 else 'FAIL'}")

# 4. Referential integrity.
foreign_keys = [
    ("orders.customer_id -> customers.customer_id",
     "orders", "customer_id", "customers", "customer_id"),
    ("order_items.order_id -> orders.order_id",
     "order_items", "order_id", "orders", "order_id"),
    ("order_items.product_id -> products.product_id",
     "order_items", "product_id", "products", "product_id"),
    ("order_items.seller_id -> sellers.seller_id",
     "order_items", "seller_id", "sellers", "seller_id"),
    ("payments.order_id -> orders.order_id",
     "payments", "order_id", "orders", "order_id"),
    ("reviews.order_id -> orders.order_id",
     "reviews", "order_id", "orders", "order_id"),
]

fk_results = {}

for label, child_name, child_key, parent_name, parent_key in foreign_keys:
    child = datasets[child_name].select(
        F.col(child_key).alias("_fk")
    ).filter(F.col("_fk").isNotNull())

    parent = (
        datasets[parent_name]
        .select(F.col(parent_key).alias("_pk"))
        .filter(F.col("_pk").isNotNull())
        .distinct()
    )

    unmatched = child.join(
        parent,
        child["_fk"] == parent["_pk"],
        "left_anti",
    ).count()

    fk_results[label] = unmatched

print("\n5. REFERENTIAL INTEGRITY")
print("-" * 90)

for label, count in fk_results.items():
    print(f"{label}: unmatched={count}, status={'PASS' if count == 0 else 'FAIL'}")

# 5. Review-data anomaly: report both datasets, do not substitute either.
print("\n6. REVIEWS PARSING ANOMALY")
print("-" * 90)

reviews = datasets["reviews"]
print("Original reviews rows:", reviews.count())
print("Original reviews schema:")
reviews.printSchema()

recovered_path = SILVER_DIR / "reviews_recovered"

if recovered_path.is_dir():
    recovered = spark.read.parquet(str(recovered_path))
    recovered_rows = recovered.count()
    print("Diagnostic recovered reviews rows:", recovered_rows)
    print("Difference from original parsed rows:",
          reviews.count() - recovered_rows)

    recovered_required = [
        "review_id", "order_id", "review_score",
        "review_creation_date", "review_answer_timestamp",
    ]
    missing_recovered = [
        c for c in recovered_required if c not in recovered.columns
    ]

    print("Recovered dataset missing required columns:", missing_recovered)

    if not missing_recovered:
        invalid_scores = recovered.filter(
            F.col("review_score").isNull() |
            (~F.col("review_score").between(1, 5))
        ).count()

        null_review_keys = recovered.filter(
            F.col("review_id").isNull() | F.col("order_id").isNull()
        ).count()

        print("Recovered invalid/null scores:", invalid_scores)
        print("Recovered null review/order keys:", null_review_keys)
else:
    print("No reviews_recovered diagnostic dataset found.")

print(
    "REVIEW STATUS: WARN — parsing discrepancy remains unresolved; "
    "do not treat the recovery candidate as authoritative."
)

# 6. Consolidated summary.
core_failures = []

for r in results:
    if r["row_count"] != "PASS" or r["schema"] != "PASS":
        core_failures.append(f"{r['dataset']}: row count/schema")

for name, r in key_results.items():
    if r["status"] != "PASS":
        core_failures.append(f"{name}: key integrity")

for name, count in field_results.items():
    if count != 0:
        core_failures.append(f"{name}: required fields")

for name, count in numeric_results.items():
    if count != 0:
        core_failures.append(f"{name}: numeric/range constraint")

for label, count in fk_results.items():
    if count != 0:
        core_failures.append(f"{label}: foreign key")

print("\n7. CONSOLIDATED SILVER RESULT")
print("-" * 90)
print("Datasets with expected row counts:",
      sum(r["row_count"] == "PASS" for r in results), "/", len(results))
print("Datasets with required schemas:",
      sum(r["schema"] == "PASS" for r in results), "/", len(results))
print("Primary/composite key checks passed:",
      sum(r["status"] == "PASS" for r in key_results.values()),
      "/", len(key_results))
print("Foreign-key checks passed:",
      sum(v == 0 for v in fk_results.values()), "/", len(fk_results))
print("Core validation failures:", core_failures if core_failures else "None")
print(
    "Overall core Silver validation:",
    "PASS" if not core_failures else "CHECK REQUIRED"
)
print("Review anomaly:", "OPEN — documented separately")

1. ROW COUNT AND SCHEMA VALIDATION
------------------------------------------------------------------------------------------
customers: rows=99441/99441, row_count=PASS, schema=PASS, missing_columns=[]
orders: rows=99441/99441, row_count=PASS, schema=PASS, missing_columns=[]
order_items: rows=112650/112650, row_count=PASS, schema=PASS, missing_columns=[]
payments: rows=103886/103886, row_count=PASS, schema=PASS, missing_columns=[]
reviews: rows=104162/104162, row_count=PASS, schema=PASS, missing_columns=[]
geolocation: rows=1000163/1000163, row_count=PASS, schema=PASS, missing_columns=[]
products: rows=32951/32951, row_count=PASS, schema=PASS, missing_columns=[]
sellers: rows=3095/3095, row_count=PASS, schema=PASS, missing_columns=[]
category_translation: rows=71/71, row_count=PASS, schema=PASS, missing_columns=[]

2. KEY INTEGRITY
------------------------------------------------------------------------------------------
customers: null_key_rows=0, duplicate_key_groups=0, status=PASS


In [60]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

SILVER_DIR = PROJECT_DIR / "data" / "silver"

spark = (
    SparkSession.builder
    .appName("DiagnoseSilverReviewIntegrity")
    .getOrCreate()
)

reviews = spark.read.parquet(str(SILVER_DIR / "reviews"))
orders = spark.read.parquet(str(SILVER_DIR / "orders"))
recovered_path = SILVER_DIR / "reviews_recovered"

order_keys = (
    orders.select("order_id")
    .filter(F.col("order_id").isNotNull())
    .distinct()
)

# Compare the original parsed review records with known order IDs.
original_unmatched = (
    reviews.filter(F.col("order_id").isNotNull())
    .join(order_keys, on="order_id", how="left_anti")
)

print("ORIGINAL REVIEWS — UNMATCHED ORDER DIAGNOSTICS")
print("-" * 65)
print("Total parsed reviews:", reviews.count())
print("Rows with null order_id:", reviews.filter(
    F.col("order_id").isNull()
).count())
print("Unmatched rows:", original_unmatched.count())
print("Distinct unmatched order IDs:",
      original_unmatched.select("order_id").distinct().count())
print("Unmatched rows with null review_id:",
      original_unmatched.filter(F.col("review_id").isNull()).count())

print("\nReview-score distribution in unmatched rows:")
original_unmatched.groupBy("review_score").count().orderBy(
    F.desc("count")
).show(20, truncate=False)

print("\nSample unmatched records:")
original_unmatched.select(
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
).show(10, truncate=60)

# Check whether the separately recovered dataset has the same problem.
if recovered_path.is_dir():
    recovered = spark.read.parquet(str(recovered_path))

    recovered_unmatched = (
        recovered.filter(F.col("order_id").isNotNull())
        .join(order_keys, on="order_id", how="left_anti")
    )

    print("\nRECOVERED REVIEWS — COMPARISON")
    print("-" * 65)
    print("Recovered rows:", recovered.count())
    print("Unmatched recovered rows:", recovered_unmatched.count())
    print("Distinct unmatched recovered order IDs:",
          recovered_unmatched.select("order_id").distinct().count())

    print("\nSample recovered unmatched records:")
    recovered_unmatched.select(
        "review_id", "order_id", "review_score"
    ).show(10, truncate=False)

ORIGINAL REVIEWS — UNMATCHED ORDER DIAGNOSTICS
-----------------------------------------------------------------
Total parsed reviews: 104162
Rows with null order_id: 2236
Unmatched rows: 2702
Distinct unmatched order IDs: 1069
Unmatched rows with null review_id: 1

Review-score distribution in unmatched rows:
+-------------------+-----+
|review_score       |count|
+-------------------+-----+
|NULL               |144  |
|2018-05-19 00:00:00|5    |
|2018-02-01 00:00:00|4    |
|2018-03-17 00:00:00|4    |
|2018-01-04 00:00:00|4    |
|2018-07-27 00:00:00|4    |
|2017-12-29 00:00:00|4    |
|2018-08-09 00:00:00|4    |
|2017-12-20 00:00:00|3    |
|2017-11-29 00:00:00|3    |
|2018-02-28 00:00:00|3    |
|2018-08-06 00:09:52|3    |
|2018-08-24 00:00:00|3    |
|2017-06-07 00:00:00|3    |
|2017-03-30 00:00:00|3    |
|2018-08-19 00:39:54|3    |
|2018-02-21 00:00:00|3    |
|2018-05-17 00:00:00|3    |
|2018-04-03 00:00:00|3    |
|2018-05-11 00:00:00|3    |
+-------------------+-----+
only showing top

Validate the recovered reviews

In [61]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

SILVER_DIR = PROJECT_DIR / "data" / "silver"

spark = (
    SparkSession.builder
    .appName("ValidateRecoveredReviews")
    .getOrCreate()
)

reviews = spark.read.parquet(str(SILVER_DIR / "reviews_recovered"))
orders = spark.read.parquet(str(SILVER_DIR / "orders"))

order_keys = orders.select("order_id").distinct()

checks = {
    "total_rows": reviews.count(),
    "null_review_id_rows": reviews.filter(
        F.col("review_id").isNull()
    ).count(),
    "null_order_id_rows": reviews.filter(
        F.col("order_id").isNull()
    ).count(),
    "duplicate_review_id_groups": (
        reviews.groupBy("review_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    ),
    "invalid_review_score_rows": reviews.filter(
        F.col("review_score").isNull()
        | (~F.col("review_score").between(1, 5))
    ).count(),
    "null_review_creation_timestamp_rows": reviews.filter(
        F.col("review_creation_date").isNull()
    ).count(),
    "null_review_answer_timestamp_rows": reviews.filter(
        F.col("review_answer_timestamp").isNull()
    ).count(),
    "unmatched_order_rows": (
        reviews.join(order_keys, on="order_id", how="left_anti").count()
    ),
}

print("RECOVERED REVIEWS QUALITY VALIDATION")
print("-" * 65)

for name, value in checks.items():
    print(f"{name}: {value}")

print("\nReview score distribution:")
reviews.groupBy("review_score").count().orderBy(
    "review_score"
).show()

print("Recovered schema:")
reviews.printSchema()

RECOVERED REVIEWS QUALITY VALIDATION
-----------------------------------------------------------------
total_rows: 99224
null_review_id_rows: 0
null_order_id_rows: 0
duplicate_review_id_groups: 789
invalid_review_score_rows: 0
null_review_creation_timestamp_rows: 0
null_review_answer_timestamp_rows: 0
unmatched_order_rows: 0

Review score distribution:
+------------+-----+
|review_score|count|
+------------+-----+
|           1|11424|
|           2| 3151|
|           3| 8179|
|           4|19142|
|           5|57328|
+------------+-----+

Recovered schema:
root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)



In [67]:
from pyspark.sql import functions as F
from pathlib import Path
import csv

raw_reviews_path = (
    PROJECT_DIR
    / "data"
    / "raw"
    / "olist"
    / "olist_order_reviews_dataset.csv"
)

print("RAW REVIEWS PARSER COMPARISON")
print("-" * 60)

# 1. Python CSV logical-record count
with open(raw_reviews_path, "r", encoding="utf-8-sig", newline="") as file:
    python_count = sum(1 for _ in csv.reader(file, strict=False))

print("Python CSV records including header:", python_count)
print("Python CSV data records:", python_count - 1)

# 2. Spark's default CSV parsing
spark_reviews_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(str(raw_reviews_path))
)

print("Spark default parsed rows:", spark_reviews_raw.count())
print("Spark parsed columns:", len(spark_reviews_raw.columns))
print("Spark column names:", spark_reviews_raw.columns)

# 3. Compare the persisted Silver datasets
reviews_path = SILVER_DIR / "reviews"
recovered_path = SILVER_DIR / "reviews_recovered"

for name, path in [
    ("Persisted Silver reviews", reviews_path),
    ("Recovered reviews candidate", recovered_path),
]:
    if path.exists():
        df = spark.read.parquet(str(path))
        print(f"{name}: {df.count()} rows, {len(df.columns)} columns")
    else:
        print(f"{name}: path not found at {path}")

RAW REVIEWS PARSER COMPARISON
------------------------------------------------------------
Python CSV records including header: 99225
Python CSV data records: 99224
Spark default parsed rows: 104162
Spark parsed columns: 7
Spark column names: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
Persisted Silver reviews: 104162 rows, 7 columns
Recovered reviews candidate: 99224 rows, 7 columns


In [74]:
from pathlib import Path

# Register persisted Silver Parquet datasets as temporary SQL views.
silver_datasets = [
    "customers",
    "orders",
    "order_items",
    "payments",
    "reviews",
    "geolocation",
    "products",
    "sellers",
    "category_translation",
]

print("SILVER DATASET REGISTRATION")
print("-" * 55)

silver_counts = {}

for dataset in silver_datasets:
    dataset_path = SILVER_DIR / dataset

    if not dataset_path.exists():
        print(f"NOT FOUND: {dataset} -> {dataset_path}")
        continue

    df = spark.read.parquet(str(dataset_path))
    df.createOrReplaceTempView(f"silver_{dataset}")

    row_count = df.count()
    silver_counts[dataset] = row_count

    print(f"Registered silver_{dataset}: {row_count:,} rows")

# Register the candidate separately for diagnostics only.
candidate_path = SILVER_DIR / "reviews_candidate"

if candidate_path.exists():
    candidate_df = spark.read.parquet(str(candidate_path))
    candidate_df.createOrReplaceTempView("silver_reviews_candidate")
    print(f"Registered silver_reviews_candidate: {candidate_df.count():,} rows")

print("\nRegistration complete. No datasets were modified.")

SILVER DATASET REGISTRATION
-------------------------------------------------------
Registered silver_customers: 99,441 rows
Registered silver_orders: 99,441 rows
Registered silver_order_items: 112,650 rows
Registered silver_payments: 103,886 rows
Registered silver_reviews: 104,162 rows
Registered silver_geolocation: 1,000,163 rows
Registered silver_products: 32,951 rows
Registered silver_sellers: 3,095 rows
Registered silver_category_translation: 71 rows
Registered silver_reviews_candidate: 99,224 rows

Registration complete. No datasets were modified.


In [75]:
from collections import OrderedDict

silver_sql_checks = OrderedDict([
    ("customers: null customer_id",
     """SELECT COUNT(*) AS n FROM silver_customers
        WHERE customer_id IS NULL"""),

    ("customers: duplicate customer_id",
     """SELECT COUNT(*) AS n FROM (
           SELECT customer_id FROM silver_customers
           GROUP BY customer_id HAVING COUNT(*) > 1
        ) t"""),

    ("orders: null order_id",
     """SELECT COUNT(*) AS n FROM silver_orders
        WHERE order_id IS NULL"""),

    ("orders: duplicate order_id",
     """SELECT COUNT(*) AS n FROM (
           SELECT order_id FROM silver_orders
           GROUP BY order_id HAVING COUNT(*) > 1
        ) t"""),

    ("order_items: null composite key",
     """SELECT COUNT(*) AS n FROM silver_order_items
        WHERE order_id IS NULL OR order_item_id IS NULL"""),

    ("order_items: duplicate composite key",
     """SELECT COUNT(*) AS n FROM (
           SELECT order_id, order_item_id FROM silver_order_items
           GROUP BY order_id, order_item_id HAVING COUNT(*) > 1
        ) t"""),

    ("products: null product_id",
     """SELECT COUNT(*) AS n FROM silver_products
        WHERE product_id IS NULL"""),

    ("products: duplicate product_id",
     """SELECT COUNT(*) AS n FROM (
           SELECT product_id FROM silver_products
           GROUP BY product_id HAVING COUNT(*) > 1
        ) t"""),

    ("sellers: null seller_id",
     """SELECT COUNT(*) AS n FROM silver_sellers
        WHERE seller_id IS NULL"""),

    ("sellers: duplicate seller_id",
     """SELECT COUNT(*) AS n FROM (
           SELECT seller_id FROM silver_sellers
           GROUP BY seller_id HAVING COUNT(*) > 1
        ) t"""),

    ("category_translation: duplicate category key",
     """SELECT COUNT(*) AS n FROM (
           SELECT product_category_name FROM silver_category_translation
           GROUP BY product_category_name HAVING COUNT(*) > 1
        ) t"""),

    ("order_items: negative price or freight",
     """SELECT COUNT(*) AS n FROM silver_order_items
        WHERE TRY_CAST(price AS DOUBLE) < 0
           OR TRY_CAST(freight_value AS DOUBLE) < 0"""),

    ("payments: negative value or installments",
     """SELECT COUNT(*) AS n FROM silver_payments
        WHERE TRY_CAST(payment_value AS DOUBLE) < 0
           OR TRY_CAST(payment_installments AS INT) < 0"""),

    ("reviews candidate: duplicate composite key",
     """SELECT COUNT(*) AS n FROM (
           SELECT review_id, order_id FROM silver_reviews_candidate
           GROUP BY review_id, order_id HAVING COUNT(*) > 1
        ) t"""),

    ("reviews candidate: invalid score",
     """SELECT COUNT(*) AS n FROM silver_reviews_candidate
        WHERE TRY_CAST(review_score AS INT) NOT BETWEEN 1 AND 5
           OR TRY_CAST(review_score AS INT) IS NULL"""),
])

print("SILVER SQL INTEGRITY CHECKS")
print("-" * 75)

results = []

for check_name, query in silver_sql_checks.items():
    count = int(spark.sql(query).first()["n"])
    status = "PASS" if count == 0 else "FAIL"
    results.append((check_name, count, status))
    print(f"{status:4} | {check_name}: {count:,}")

print("-" * 75)
print("Checks:", len(results))
print("Passed:", sum(status == "PASS" for _, _, status in results))
print("Failed:", sum(status == "FAIL" for _, _, status in results))

SILVER SQL INTEGRITY CHECKS
---------------------------------------------------------------------------
PASS | customers: null customer_id: 0
PASS | customers: duplicate customer_id: 0
PASS | orders: null order_id: 0
PASS | orders: duplicate order_id: 0
PASS | order_items: null composite key: 0
PASS | order_items: duplicate composite key: 0
PASS | products: null product_id: 0
PASS | products: duplicate product_id: 0
PASS | sellers: null seller_id: 0
PASS | sellers: duplicate seller_id: 0
PASS | category_translation: duplicate category key: 0
PASS | order_items: negative price or freight: 0
PASS | payments: negative value or installments: 0
PASS | reviews candidate: duplicate composite key: 0
PASS | reviews candidate: invalid score: 0
---------------------------------------------------------------------------
Checks: 15
Passed: 15
Failed: 0


In [76]:
from collections import OrderedDict

foreign_key_checks = OrderedDict([
    (
        "Orders -> Customers: unmatched customer IDs",
        """SELECT COUNT(*) AS n
           FROM silver_orders o
           LEFT ANTI JOIN silver_customers c
           ON o.customer_id = c.customer_id"""
    ),
    (
        "Order items -> Orders: unmatched order IDs",
        """SELECT COUNT(*) AS n
           FROM silver_order_items i
           LEFT ANTI JOIN silver_orders o
           ON i.order_id = o.order_id"""
    ),
    (
        "Order items -> Products: unmatched product IDs",
        """SELECT COUNT(*) AS n
           FROM silver_order_items i
           LEFT ANTI JOIN silver_products p
           ON i.product_id = p.product_id"""
    ),
    (
        "Order items -> Sellers: unmatched seller IDs",
        """SELECT COUNT(*) AS n
           FROM silver_order_items i
           LEFT ANTI JOIN silver_sellers s
           ON i.seller_id = s.seller_id"""
    ),
    (
        "Payments -> Orders: unmatched order IDs",
        """SELECT COUNT(*) AS n
           FROM silver_payments p
           LEFT ANTI JOIN silver_orders o
           ON p.order_id = o.order_id"""
    ),
    (
        "Original reviews -> Orders: unmatched order IDs",
        """SELECT COUNT(*) AS n
           FROM silver_reviews r
           LEFT ANTI JOIN silver_orders o
           ON r.order_id = o.order_id"""
    ),
    (
        "Review candidate -> Orders: unmatched order IDs",
        """SELECT COUNT(*) AS n
           FROM silver_reviews_candidate r
           LEFT ANTI JOIN silver_orders o
           ON r.order_id = o.order_id"""
    ),
])

print("SILVER FOREIGN-KEY VALIDATION")
print("-" * 80)

for check_name, query in foreign_key_checks.items():
    count = int(spark.sql(query).first()["n"])

    if check_name.startswith("Original reviews"):
        status = "KNOWN EXCEPTION" if count > 0 else "PASS"
    else:
        status = "PASS" if count == 0 else "FAIL"

    print(f"{status:15} | {check_name}: {count:,}")

print("-" * 80)
print("Validation complete. No datasets were modified.")

SILVER FOREIGN-KEY VALIDATION
--------------------------------------------------------------------------------
PASS            | Orders -> Customers: unmatched customer IDs: 0
PASS            | Order items -> Orders: unmatched order IDs: 0
PASS            | Order items -> Products: unmatched product IDs: 0
PASS            | Order items -> Sellers: unmatched seller IDs: 0
PASS            | Payments -> Orders: unmatched order IDs: 0
KNOWN EXCEPTION | Original reviews -> Orders: unmatched order IDs: 4,938
PASS            | Review candidate -> Orders: unmatched order IDs: 0
--------------------------------------------------------------------------------
Validation complete. No datasets were modified.


In [73]:
from pathlib import Path
from pyspark.sql import functions as F

SILVER_EXPECTED_COUNTS = {
    "customers": 99_441,
    "orders": 99_441,
    "order_items": 112_650,
    "payments": 103_886,
    "reviews": 104_162,  # Existing persisted dataset; anomaly documented separately
    "geolocation": 1_000_163,
    "products": 32_951,
    "sellers": 3_095,
    "category_translation": 71,
}

print("SILVER LAYER: SCHEMA AND ROW-COUNT VALIDATION")
print("-" * 75)

silver_results = []

for dataset, expected_rows in SILVER_EXPECTED_COUNTS.items():
    dataset_path = SILVER_DIR / dataset

    if not dataset_path.exists():
        silver_results.append(
            (dataset, expected_rows, None, None, "MISSING")
        )
        continue

    df = spark.read.parquet(str(dataset_path))
    actual_rows = df.count()
    column_count = len(df.columns)

    status = "PASS" if actual_rows == expected_rows else "CHECK REQUIRED"

    silver_results.append(
        (dataset, expected_rows, actual_rows, column_count, status)
    )

for dataset, expected, actual, columns, status in silver_results:
    print(
        f"{dataset:24} | expected={expected:>9,} | "
        f"actual={actual if actual is not None else 'N/A':>9} | "
        f"columns={columns if columns is not None else 'N/A':>3} | "
        f"{status}"
    )

passed = sum(result[4] == "PASS" for result in silver_results)
print("-" * 75)
print(f"Row-count checks passed: {passed}/{len(SILVER_EXPECTED_COUNTS)}")

SILVER LAYER: SCHEMA AND ROW-COUNT VALIDATION
---------------------------------------------------------------------------
customers                | expected=   99,441 | actual=    99441 | columns=  5 | PASS
orders                   | expected=   99,441 | actual=    99441 | columns=  8 | PASS
order_items              | expected=  112,650 | actual=   112650 | columns=  7 | PASS
payments                 | expected=  103,886 | actual=   103886 | columns=  5 | PASS
reviews                  | expected=  104,162 | actual=   104162 | columns=  7 | PASS
geolocation              | expected=1,000,163 | actual=  1000163 | columns=  5 | PASS
products                 | expected=   32,951 | actual=    32951 | columns=  9 | PASS
sellers                  | expected=    3,095 | actual=     3095 | columns=  4 | PASS
category_translation     | expected=       71 | actual=       71 | columns=  2 | PASS
---------------------------------------------------------------------------
Row-count checks passed: 9/9

Gold validation

In [58]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

DATA_DIR = PROJECT_DIR / "data"
SILVER_DIR = DATA_DIR / "silver"
PROCESSED_DIR = DATA_DIR / "processed"
GOLD_DIMENSIONAL_DIR = DATA_DIR / "gold" / "dimensional"

spark = (
    SparkSession.builder
    .appName("SalesAnalyticsGoldSQLValidation")
    .getOrCreate()
)

# Read persisted Gold tables
fact = spark.read.parquet(
    str(GOLD_DIMENSIONAL_DIR / "fact_sales")
)
dim_order = spark.read.parquet(
    str(GOLD_DIMENSIONAL_DIR / "dim_order")
)
dim_customer = spark.read.parquet(
    str(GOLD_DIMENSIONAL_DIR / "dim_customer")
)
dim_product = spark.read.parquet(
    str(GOLD_DIMENSIONAL_DIR / "dim_product")
)
dim_seller = spark.read.parquet(
    str(GOLD_DIMENSIONAL_DIR / "dim_seller")
)

# Read the existing Phase 1 monthly analytics CSV.
monthly = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(PROCESSED_DIR / "monthly_sales_analytics.csv"))
)

# Register SQL views.
fact.createOrReplaceTempView("gold_fact_sales")
dim_order.createOrReplaceTempView("gold_dim_order")
dim_customer.createOrReplaceTempView("gold_dim_customer")
dim_product.createOrReplaceTempView("gold_dim_product")
dim_seller.createOrReplaceTempView("gold_dim_seller")
monthly.createOrReplaceTempView("phase1_monthly_sales")

# 1. Fact integrity.
integrity = spark.sql("""
    SELECT
        COUNT(*) AS fact_rows,
        COUNT(DISTINCT CONCAT(order_id, '|', CAST(order_item_id AS STRING)))
            AS distinct_order_item_keys,
        SUM(quantity) AS total_quantity,
        SUM(CASE WHEN order_id IS NULL
                      OR order_item_id IS NULL
                 THEN 1 ELSE 0 END) AS null_business_keys,
        SUM(CASE WHEN dim_order_key IS NULL
                      OR dim_customer_key IS NULL
                      OR dim_product_key IS NULL
                      OR dim_seller_key IS NULL
                 THEN 1 ELSE 0 END) AS null_dimension_keys,
        SUM(price) AS item_price_total,
        SUM(freight_value) AS freight_total
    FROM gold_fact_sales
""").first().asDict()

# 2. Verify each foreign key resolves to its dimension.
fk_checks = {
    "order": spark.sql("""
        SELECT COUNT(*) AS n
        FROM gold_fact_sales f
        LEFT ANTI JOIN gold_dim_order d
          ON f.dim_order_key = d.dim_order_key
    """).first()["n"],

    "customer": spark.sql("""
        SELECT COUNT(*) AS n
        FROM gold_fact_sales f
        LEFT ANTI JOIN gold_dim_customer d
          ON f.dim_customer_key = d.dim_customer_key
    """).first()["n"],

    "product": spark.sql("""
        SELECT COUNT(*) AS n
        FROM gold_fact_sales f
        LEFT ANTI JOIN gold_dim_product d
          ON f.dim_product_key = d.dim_product_key
    """).first()["n"],

    "seller": spark.sql("""
        SELECT COUNT(*) AS n
        FROM gold_fact_sales f
        LEFT ANTI JOIN gold_dim_seller d
          ON f.dim_seller_key = d.dim_seller_key
    """).first()["n"],
}

# 3. Reconcile eligible sales against Phase 1 monthly analytics.
# Canceled and unavailable orders are excluded only for this comparison.
gold_eligible = spark.sql("""
    SELECT
        COUNT(*) AS eligible_fact_rows,
        SUM(f.quantity) AS eligible_quantity,
        SUM(f.price) AS eligible_item_revenue,
        SUM(f.freight_value) AS eligible_freight
    FROM gold_fact_sales f
    INNER JOIN gold_dim_order o
      ON f.dim_order_key = o.dim_order_key
    WHERE LOWER(o.order_status) NOT IN ('canceled', 'unavailable')
""").first().asDict()

phase1_totals = spark.sql("""
    SELECT
        SUM(total_items) AS phase1_quantity,
        SUM(item_revenue) AS phase1_item_revenue,
        SUM(freight_cost) AS phase1_freight
    FROM phase1_monthly_sales
""").first().asDict()

def close_enough(a, b, tolerance=0.01):
    if a is None or b is None:
        return a is None and b is None
    return abs(float(a) - float(b)) <= tolerance

checks = {
    "fact_rows_equal_distinct_order_item_keys":
        integrity["fact_rows"] == integrity["distinct_order_item_keys"],

    "no_null_business_keys":
        integrity["null_business_keys"] == 0,

    "no_null_dimension_keys":
        integrity["null_dimension_keys"] == 0,

    "all_order_references_resolve": fk_checks["order"] == 0,
    "all_customer_references_resolve": fk_checks["customer"] == 0,
    "all_product_references_resolve": fk_checks["product"] == 0,
    "all_seller_references_resolve": fk_checks["seller"] == 0,

    "eligible_quantity_matches_phase1":
        close_enough(
            gold_eligible["eligible_quantity"],
            phase1_totals["phase1_quantity"]
        ),

    "eligible_revenue_matches_phase1":
        close_enough(
            gold_eligible["eligible_item_revenue"],
            phase1_totals["phase1_item_revenue"]
        ),

    "eligible_freight_matches_phase1":
        close_enough(
            gold_eligible["eligible_freight"],
            phase1_totals["phase1_freight"]
        ),
}

print("GOLD SQL VALIDATION")
print("-" * 70)
print("Fact rows:", integrity["fact_rows"])
print("Distinct order-item keys:", integrity["distinct_order_item_keys"])
print("Total quantity, all statuses:", integrity["total_quantity"])
print("Null business keys:", integrity["null_business_keys"])
print("Null dimension keys:", integrity["null_dimension_keys"])
print("Unmatched dimension references:", fk_checks)
print()

print("ALL-STATUS FACT TOTALS")
print("Item price:", integrity["item_price_total"])
print("Freight:", integrity["freight_total"])
print()

print("ELIGIBLE SALES RECONCILIATION")
print("Eligible fact rows:", gold_eligible["eligible_fact_rows"])
print("Gold eligible quantity:", gold_eligible["eligible_quantity"])
print("Phase 1 quantity:", phase1_totals["phase1_quantity"])
print("Gold eligible revenue:", gold_eligible["eligible_item_revenue"])
print("Phase 1 revenue:", phase1_totals["phase1_item_revenue"])
print("Gold eligible freight:", gold_eligible["eligible_freight"])
print("Phase 1 freight:", phase1_totals["phase1_freight"])
print()

print("VALIDATION CHECKS")
for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

# Status distribution is informational; no status is filtered out here.
print("\nORDER STATUS DISTRIBUTION")
spark.sql("""
    SELECT o.order_status,
           COUNT(*) AS item_rows,
           SUM(f.price) AS item_revenue,
           SUM(f.freight_value) AS freight
    FROM gold_fact_sales f
    JOIN gold_dim_order o
      ON f.dim_order_key = o.dim_order_key
    GROUP BY o.order_status
    ORDER BY item_rows DESC
""").show(truncate=False)

print("Overall:", "PASS" if all(checks.values()) else "CHECK REQUIRED")

GOLD SQL VALIDATION
----------------------------------------------------------------------
Fact rows: 112650
Distinct order-item keys: 112650
Total quantity, all statuses: 112650
Null business keys: 0
Null dimension keys: 0
Unmatched dimension references: {'order': 0, 'customer': 0, 'product': 0, 'seller': 0}

ALL-STATUS FACT TOTALS
Item price: 13591643.700007346
Freight: 2251909.5399999404

ELIGIBLE SALES RECONCILIATION
Eligible fact rows: 112101
Gold eligible quantity: 112101
Phase 1 quantity: 112101.0
Gold eligible revenue: 13494400.74001409
Phase 1 revenue: 13494400.739999998
Gold eligible freight: 2241126.290000232
Phase 1 freight: 2241126.29

VALIDATION CHECKS
fact_rows_equal_distinct_order_item_keys: PASS
no_null_business_keys: PASS
no_null_dimension_keys: PASS
all_order_references_resolve: PASS
all_customer_references_resolve: PASS
all_product_references_resolve: PASS
all_seller_references_resolve: PASS
eligible_quantity_matches_phase1: PASS
eligible_revenue_matches_phase1: PAS

Persistant test

In [77]:
from pathlib import Path

test_path = PROJECT_DIR / "tests" / "test_pipeline.py"

print(f"Test file: {test_path.relative_to(PROJECT_DIR)}")
print("-" * 70)

with open(test_path, "r", encoding="utf-8") as file:
    print(file.read())

Test file: tests/test_pipeline.py
----------------------------------------------------------------------
"""Tests for sales analytics pipeline outputs."""

import unittest
from pathlib import Path
import pandas as pd


REPO_ROOT = Path(__file__).resolve().parents[1]
PROCESSED_DIR = REPO_ROOT / "data" / "processed"


class TestPipelineOutputs(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.orders = pd.read_csv(
            PROCESSED_DIR / "order_analytics.csv"
        )
        cls.products = pd.read_csv(
            PROCESSED_DIR / "product_analytics.csv"
        )
        cls.categories = pd.read_csv(
            PROCESSED_DIR / "category_analytics.csv"
        )
        cls.customers = pd.read_csv(
            PROCESSED_DIR / "customer_analytics.csv"
        )
        cls.monthly = pd.read_csv(
            PROCESSED_DIR / "monthly_sales_analytics.csv"
        )
        cls.status = pd.read_csv(
            PROCESSED_DIR / "order_status_summary.csv"
        

Run the existing test suite

In [80]:
%pip install "psycopg[binary]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.5/215.5 kB 19.0 MB/s eta 0:00:00


In [82]:
import os
import subprocess
import sys

src_dir = str(PROJECT_DIR / "src")

env = os.environ.copy()
env["PYTHONPATH"] = (
    src_dir
    if not env.get("PYTHONPATH")
    else src_dir + os.pathsep + env["PYTHONPATH"]
)

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "discover",
        "-s",
        str(PROJECT_DIR / "tests"),
        "-p",
        "test_*.py",
        "-v",
    ],
    cwd=str(PROJECT_DIR),
    env=env,
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

print("Exit code:", result.returncode)

[0.003s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.003s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
SALES ANALYTICS DATA PIPELINE

[1/5] Validating source data...

[2/5] Building order analytics...

[3/5] Building product and category analytics...

[4/5] Building customer analytics...

[5/5] Building sales performance analytics...

PIPELINE COMPLETED SUCCESSFULLY
Duration: 0.0 seconds
Validation checks: 1
Order rows: 2
Product rows: 1
Cat

# Documentation

In [83]:
from pathlib import Path

for folder_name in ["docs", "tests", "sql"]:
    folder = PROJECT_DIR / folder_name
    print(f"\n{folder_name.upper()}")
    print("-" * 50)

    if folder.exists():
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                print(path.relative_to(PROJECT_DIR))
    else:
        print("Directory not found")


DOCS
--------------------------------------------------
docs/phase-3-pyspark-transformation.md

TESTS
--------------------------------------------------
tests/__init__.py
tests/__pycache__/__init__.cpython-313.pyc
tests/__pycache__/test_pipeline.cpython-313.pyc
tests/__pycache__/test_spark_transformations.cpython-313.pyc
tests/test_pipeline.py
tests/test_spark_transformations.py

SQL
--------------------------------------------------
sql/01_monthly_revenue_growth.sql
sql/02_order_sales_performance.sql
sql/03_customer_repeat_analysis.sql
sql/04_top_selling_products.sql
sql/05_category_revenue_analysis.sql
sql/06_delivery_performance.sql
sql/07_top_customers_by_revenue.sql
sql/08_seller_coverage_analysis.sql


In [84]:
from pathlib import Path

print("PROJECT_DIR:", PROJECT_DIR)
print("Gold analytics exists:", GOLD_ANALYTICS_DIR.exists())
print("Gold dimensional exists:", GOLD_DIMENSIONAL_DIR.exists())

print("\nGold analytics tables:")
for name in [
    "order_analytics",
    "product_analytics",
    "category_analytics",
    "customer_analytics",
    "monthly_sales_analytics",
    "order_status_summary",
]:
    path = GOLD_ANALYTICS_DIR / name
    print(f"{name}: {'FOUND' if path.exists() else 'MISSING'}")

print("\nGold dimensional tables:")
for name in [
    "dim_customer",
    "dim_product",
    "dim_seller",
    "dim_order",
    "fact_sales",
]:
    path = GOLD_DIMENSIONAL_DIR / name
    print(f"{name}: {'FOUND' if path.exists() else 'MISSING'}")

PROJECT_DIR: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline
Gold analytics exists: True
Gold dimensional exists: True

Gold analytics tables:
order_analytics: FOUND
product_analytics: FOUND
category_analytics: FOUND
customer_analytics: FOUND
monthly_sales_analytics: FOUND
order_status_summary: FOUND

Gold dimensional tables:
dim_customer: FOUND
dim_product: FOUND
dim_seller: FOUND
dim_order: FOUND
fact_sales: FOUND


Create *test_phase4_storage_modeling.py*

In [85]:
from pathlib import Path

test_file = PROJECT_DIR / "tests" / "test_phase4_storage_modeling.py"

content = r'''"""Persistent Phase 4 tests for Parquet storage and dimensional modeling."""

import unittest
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


REPO_ROOT = Path(__file__).resolve().parents[1]
DATA_DIR = REPO_ROOT / "data"
ANALYTICS_DIR = DATA_DIR / "gold" / "analytics"
DIMENSIONAL_DIR = DATA_DIR / "gold" / "dimensional"


class TestPhase4StorageAndModeling(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        required_paths = [
            ANALYTICS_DIR / "monthly_sales_analytics",
            DIMENSIONAL_DIR / "dim_customer",
            DIMENSIONAL_DIR / "dim_product",
            DIMENSIONAL_DIR / "dim_seller",
            DIMENSIONAL_DIR / "dim_order",
            DIMENSIONAL_DIR / "fact_sales",
        ]

        missing = [str(p) for p in required_paths if not p.exists()]
        if missing:
            raise unittest.SkipTest(
                "Phase 4 Parquet data unavailable: " + ", ".join(missing)
            )

        cls.spark = (
            SparkSession.builder
            .master("local[2]")
            .appName("SalesPipelinePhase4Tests")
            .config("spark.ui.enabled", "false")
            .getOrCreate()
        )
        cls.spark.sparkContext.setLogLevel("ERROR")

        cls.analytics = {}
        for name in [
            "order_analytics",
            "product_analytics",
            "category_analytics",
            "customer_analytics",
            "monthly_sales_analytics",
            "order_status_summary",
        ]:
            path = ANALYTICS_DIR / name
            if path.exists():
                cls.analytics[name] = cls.spark.read.parquet(str(path))

        cls.tables = {}
        for name in [
            "dim_customer",
            "dim_product",
            "dim_seller",
            "dim_order",
            "fact_sales",
        ]:
            cls.tables[name] = cls.spark.read.parquet(
                str(DIMENSIONAL_DIR / name)
            )

    @classmethod
    def tearDownClass(cls):
        if hasattr(cls, "spark"):
            cls.spark.stop()

    def test_gold_analytics_row_counts(self):
        expected = {
            "order_analytics": 99441,
            "product_analytics": 32729,
            "category_analytics": 72,
            "customer_analytics": 94990,
            "monthly_sales_analytics": 24,
            "order_status_summary": 8,
        }

        for name, expected_count in expected.items():
            with self.subTest(table=name):
                self.assertIn(name, self.analytics)
                self.assertEqual(
                    self.analytics[name].count(), expected_count
                )

    def test_dimensional_table_row_counts(self):
        expected = {
            "dim_customer": 99441,
            "dim_product": 32951,
            "dim_seller": 3095,
            "dim_order": 99441,
            "fact_sales": 112650,
        }

        for name, expected_count in expected.items():
            with self.subTest(table=name):
                self.assertEqual(self.tables[name].count(), expected_count)

    def test_dimension_business_keys_are_unique_and_non_null(self):
        keys = {
            "dim_customer": "customer_id",
            "dim_product": "product_id",
            "dim_seller": "seller_id",
            "dim_order": "order_id",
        }

        for table_name, key in keys.items():
            with self.subTest(table=table_name):
                df = self.tables[table_name]
                self.assertEqual(
                    df.filter(F.col(key).isNull()).count(), 0
                )
                duplicates = (
                    df.groupBy(key)
                    .count()
                    .filter(F.col("count") > 1)
                    .count()
                )
                self.assertEqual(duplicates, 0)

    def test_fact_business_key_is_unique_and_required_keys_non_null(self):
        fact = self.tables["fact_sales"]
        required = [
            "order_id",
            "order_item_id",
            "dim_order_key",
            "dim_customer_key",
            "dim_product_key",
            "dim_seller_key",
        ]

        null_condition = None
        for column in required:
            condition = F.col(column).isNull()
            null_condition = (
                condition if null_condition is None
                else null_condition | condition
            )

        self.assertEqual(fact.filter(null_condition).count(), 0)

        duplicate_keys = (
            fact.groupBy("order_id", "order_item_id")
            .count()
            .filter(F.col("count") > 1)
            .count()
        )
        self.assertEqual(duplicate_keys, 0)

    def test_fact_foreign_keys_resolve_to_dimensions(self):
        fact = self.tables["fact_sales"]
        relationships = [
            ("dim_order_key", "dim_order"),
            ("dim_customer_key", "dim_customer"),
            ("dim_product_key", "dim_product"),
            ("dim_seller_key", "dim_seller"),
        ]

        for fact_key, dimension_name in relationships:
            with self.subTest(fact_key=fact_key):
                dimension_keys = self.tables[dimension_name].select(
                    fact_key
                ).distinct()

                unmatched = (
                    fact.select(fact_key)
                    .distinct()
                    .join(dimension_keys, on=fact_key, how="left_anti")
                )
                self.assertEqual(unmatched.count(), 0)

    def test_eligible_fact_sales_reconcile_with_monthly_analytics(self):
        fact = self.tables["fact_sales"]
        orders = self.tables["dim_order"]
        monthly = self.analytics["monthly_sales_analytics"]

        eligible = (
            fact.join(
                orders.select("order_id", "order_status"),
                on="order_id",
                how="inner",
            )
            .filter(~F.col("order_status").isin("canceled", "unavailable"))
        )

        fact_metrics = eligible.agg(
            F.count(F.lit(1)).alias("item_rows"),
            F.sum("price").alias("revenue"),
            F.sum("freight_value").alias("freight"),
        ).first()

        monthly_metrics = monthly.agg(
            F.sum("item_count").alias("item_rows"),
            F.sum("item_revenue").alias("revenue"),
            F.sum("freight_cost").alias("freight"),
        ).first()

        self.assertEqual(int(fact_metrics["item_rows"]), 112101)
        self.assertEqual(int(monthly_metrics["item_rows"]), 112101)

        self.assertAlmostEqual(
            float(fact_metrics["revenue"]),
            float(monthly_metrics["revenue"]),
            delta=0.01,
        )
        self.assertAlmostEqual(
            float(fact_metrics["freight"]),
            float(monthly_metrics["freight"]),
            delta=0.01,
        )


if __name__ == "__main__":
    unittest.main()
'''

test_file.write_text(content, encoding="utf-8")

print("Created:", test_file.relative_to(PROJECT_DIR))
print("File size:", test_file.stat().st_size, "bytes")
print("Existing tests and data were not modified.")

Created: tests/test_phase4_storage_modeling.py
File size: 6925 bytes
Existing tests and data were not modified.


run the Phase 4 tests independently

In [88]:
import os
import subprocess
import sys

env = os.environ.copy()
src_dir = str(PROJECT_DIR / "src")
env["PYTHONPATH"] = (
    src_dir
    if not env.get("PYTHONPATH")
    else src_dir + os.pathsep + env["PYTHONPATH"]
)

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "discover",
        "-s",
        str(PROJECT_DIR / "tests"),
        "-p",
        "test_phase4_storage_modeling.py",
        "-v",
    ],
    cwd=str(PROJECT_DIR),
    env=env,
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

print("Exit code:", result.returncode)

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/17 07:01:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes wher

Fixing the test error


In [87]:
from pathlib import Path

test_path = PROJECT_DIR / "tests" / "test_phase4_storage_modeling.py"
source = test_path.read_text(encoding="utf-8")

old = 'F.sum("item_count").alias("item_rows"),'
new = 'F.sum("total_items").alias("item_rows"),'

if old not in source:
    raise ValueError(
        "Expected test expression not found. No file changes were made."
    )

source = source.replace(old, new, 1)
test_path.write_text(source, encoding="utf-8")

print("Updated the monthly analytics column reference: item_count -> total_items")
print("No production code or data was modified.")

Updated the monthly analytics column reference: item_count -> total_items
No production code or data was modified.


Create documentation

In [89]:
from pathlib import Path

docs_dir = PROJECT_DIR / "docs"
docs_dir.mkdir(parents=True, exist_ok=True)

doc_path = docs_dir / "phase-4-storage-and-data-modeling.md"

content = r'''# Phase 4 — Storage and Data Modeling

## 1. Objective

Phase 4 establishes persistent Bronze, Silver, and Gold data layers using Parquet and introduces a dimensional model for sales analytics. The goal is to make processed datasets reloadable, queryable, and reconcilable against established analytics outputs.

## 2. Architecture

- **Bronze:** Raw Olist source datasets persisted as Parquet snapshots.
- **Silver:** Structured source datasets persisted as Parquet, with data-quality and relationship checks.
- **Gold analytics:** Existing Phase 1 analytics outputs persisted as Parquet.
- **Gold dimensional:** Fact and dimension tables designed for analytical queries.
- **PostgreSQL:** Existing Phase 2 relational warehouse retained separately; Phase 4 does not replace its tables.

All persisted datasets reside under the project's `data/` directory. Data files are excluded from Git by the existing `.gitignore` policy.

## 3. Persisted dataset inventory

### Bronze source snapshots

| Dataset | Rows |
|---|---:|
| customers | 99,441 |
| orders | 99,441 |
| order_items | 112,650 |
| payments | 103,886 |
| reviews | 104,162 in the original Spark parse |
| geolocation | 1,000,163 |
| products | 32,951 |
| sellers | 3,095 |
| category_translation | 71 |

### Gold analytics tables

| Table | Grain | Rows |
|---|---|---:|
| order_analytics | One row per order | 99,441 |
| product_analytics | Product/category aggregate | 32,729 |
| category_analytics | Product category aggregate | 72 |
| customer_analytics | Customer identity aggregate | 94,990 |
| monthly_sales_analytics | One row per month | 24 |
| order_status_summary | One row per order status | 8 |

### Gold dimensional tables

| Table | Grain | Rows |
|---|---|---:|
| dim_customer | One row per customer_id | 99,441 |
| dim_product | One row per product_id | 32,951 |
| dim_seller | One row per seller_id | 3,095 |
| dim_order | One row per order_id | 99,441 |
| fact_sales | One row per order item | 112,650 |

The fact table retains all order statuses. Reconciliation to the Phase 1 sales analytics excludes `canceled` and `unavailable` orders according to the established sales transformation logic.

## 4. Fact and dimension design

### fact_sales

The fact table contains the order-item grain, including `order_id`, `order_item_id`, dimension surrogate keys, `price`, `freight_value`, and shipping-limit date.

The composite business key is `(order_id, order_item_id)`. Revenue and freight are measured at the order-item grain.

### Dimensions

- `dim_customer` uses `customer_id` as its business key and retains `customer_unique_id` as an identity attribute. This avoids silently merging source customer records that share a `customer_unique_id`.
- `dim_product` uses `product_id`.
- `dim_seller` uses `seller_id`.
- `dim_order` uses `order_id`.

Surrogate keys are deterministic for a fixed source snapshot, but may change if new business keys are inserted earlier in the ordering. They should not be treated as permanently stable identifiers across independently rebuilt snapshots.

## 5. Validation evidence

### Silver SQL integrity

Fifteen SQL integrity checks passed, covering required keys, duplicate business keys, order-item composite-key uniqueness, non-negative item and payment values, category translation uniqueness, and review-candidate constraints.

### Silver relationships

The following relationships passed with zero unmatched records:

- orders → customers
- order_items → orders
- order_items → products
- order_items → sellers
- payments → orders

The original reviews dataset has a documented parsing exception described below. The separate multiline-parsed review candidate passed the tested composite-key, score-range, and order-reference checks.

### Gold SQL validation

The previously executed Gold SQL validation passed all 10 checks, including fact-table grain, required keys, dimension references, and sales metric reconciliation.

The eligible fact dataset contains 112,101 order items. Revenue and freight reconcile with Phase 1 monthly sales analytics within a tolerance of 0.01.

### Automated tests

The existing test suite passed 14 tests after the test environment was configured with the declared PostgreSQL driver and the repository `src` directory on `PYTHONPATH`.

The new Phase 4 test file passed 6 tests covering persisted Gold row counts, dimension keys, fact grain, fact-to-dimension references, and monthly sales reconciliation.

## 6. Known exception: reviews CSV parsing

The original persisted Silver reviews dataset contains 104,162 rows and produces 4,938 unmatched order references in the current relationship check.

An independent multiline CSV parse produces 99,224 logical records with seven columns, no null review/order IDs, valid scores from 1 to 5, no duplicate `(review_id, order_id)` pairs, and zero unmatched order references.

The raw-source audit found repeated `review_id` values, so `review_id` alone is not a valid uniqueness constraint for this source. No exact duplicate full rows were found in the 99,224-record candidate.

The 4,938-row discrepancy between the original Spark parse and the multiline candidate remains documented and unresolved. The candidate is retained separately; the original persisted Silver dataset and ingestion modules were not overwritten or changed.

## 7. Scope and limitations

- Existing Phase 1, Phase 2, and Phase 3 modules and outputs were preserved.
- Existing PostgreSQL warehouse tables were not replaced.
- The reviews parsing discrepancy remains a documented exception rather than being silently repaired.
- Phase 4 has not yet been committed or pushed to Git.
- Final full-suite verification and Git review are required before considering Phase 4 ready for delivery.

## 8. Acceptance summary

| Requirement | Status |
|---|---|
| Bronze, Silver, and Gold layers persisted | Implemented |
| Parquet outputs reload successfully | Validated |
| Sales fact and dimensions implemented | Validated |
| Gold SQL reconciliation | 10 checks passed |
| Silver SQL integrity | 15 checks passed |
| Phase 4 automated tests | 6 tests passed |
| Reviews parsing discrepancy documented | Yes; unresolved |
| Final full-suite run and Git review | Pending |
'''

doc_path.write_text(content, encoding="utf-8")

print("Created:", doc_path.relative_to(PROJECT_DIR))
print("Bytes:", doc_path.stat().st_size)
print("Existing documentation and pipeline files were not modified.")

Created: docs/phase-4-storage-and-data-modeling.md
Bytes: 6246
Existing documentation and pipeline files were not modified.


Run the complete test suite

In [90]:
import os
import subprocess
import sys

env = os.environ.copy()
src_dir = str(PROJECT_DIR / "src")
env["PYTHONPATH"] = (
    src_dir
    if not env.get("PYTHONPATH")
    else src_dir + os.pathsep + env["PYTHONPATH"]
)

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "discover",
        "-s",
        str(PROJECT_DIR / "tests"),
        "-p",
        "test_*.py",
        "-v",
    ],
    cwd=str(PROJECT_DIR),
    env=env,
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

print("Exit code:", result.returncode)

[0.004s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.004s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
SALES ANALYTICS DATA PIPELINE

[1/5] Validating source data...

[2/5] Building order analytics...

[3/5] Building product and category analytics...

[4/5] Building customer analytics...

[5/5] Building sales performance analytics...

PIPELINE COMPLETED SUCCESSFULLY
Duration: 0.01 seconds
Validation checks: 1
Order rows: 2
Product rows: 1
Ca

# Push to Github

In [91]:
import subprocess

commands = [
    ["git", "status", "--short"],
    ["git", "diff", "--stat"],
    ["git", "diff", "--cached", "--stat"],
    ["git", "diff", "--check"],
]

for cmd in commands:
    print(f"\n$ {' '.join(cmd)}")
    result = subprocess.run(
        cmd,
        cwd=str(PROJECT_DIR),
        capture_output=True,
        text=True
    )
    print(result.stdout if result.stdout else "(no output)")
    if result.stderr:
        print(result.stderr)
    print("Exit code:", result.returncode)

print("\nUntracked files:")
result = subprocess.run(
    ["git", "ls-files", "--others", "--exclude-standard"],
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True
)
print(result.stdout if result.stdout else "(none)")


$ git status --short
 M 02_postgresql_data_warehouse.ipynb
 M 03_PySpark_transformation.ipynb
?? 04_Storage_and_Data_modeling.ipynb
?? docs/phase-4-storage-and-data-modeling.md
?? tests/test_phase4_storage_modeling.py

Exit code: 0

$ git diff --stat
 02_postgresql_data_warehouse.ipynb | 2 +-
 03_PySpark_transformation.ipynb    | 2 +-
 2 files changed, 2 insertions(+), 2 deletions(-)

Exit code: 0

$ git diff --cached --stat
(no output)
Exit code: 0

$ git diff --check
(no output)
Exit code: 0

Untracked files:
04_Storage_and_Data_modeling.ipynb
docs/phase-4-storage-and-data-modeling.md
tests/test_phase4_storage_modeling.py



In [92]:
import subprocess

for notebook in [
    "02_postgresql_data_warehouse.ipynb",
    "03_PySpark_transformation.ipynb",
]:
    print(f"\n--- {notebook} ---")
    result = subprocess.run(
        ["git", "diff", "--numstat", "--", notebook],
        cwd=str(PROJECT_DIR),
        capture_output=True,
        text=True
    )
    print(result.stdout or "(no diff summary)")

    result = subprocess.run(
        ["git", "diff", "--word-diff=porcelain", "--", notebook],
        cwd=str(PROJECT_DIR),
        capture_output=True,
        text=True
    )
    print(result.stdout[:5000] or "(no content diff)")
    if result.stderr:
        print(result.stderr)


--- 02_postgresql_data_warehouse.ipynb ---
1	1	02_postgresql_data_warehouse.ipynb

diff --git a/02_postgresql_data_warehouse.ipynb b/02_postgresql_data_warehouse.ipynb
index ee56bcc..7c758d6 100644
--- a/02_postgresql_data_warehouse.ipynb
+++ b/02_postgresql_data_warehouse.ipynb
@@ -1 +1 @@
-{"nbformat":4,"nbformat_minor":0,"metadata":{"colab":{"provenance":[],"authorship_tag":"ABX9TyOM8HvkweKVT5tezhnPF2iH"},"kernelspec":{"name":"python3","display_name":"Python 3"},"language_info":{"name":"python"}},"cells":[{"cell_type":"code","execution_count":2,"metadata":{"colab":{"base_uri":"https://localhost:8080/"},"id":"nzu8PchbUAQD","executionInfo":{"status":"ok","timestamp":1792086552188,"user_tz":-330,"elapsed":7098,"user":{"displayName":"Vipul Kumawat","userId":"10244865491502480161"}},"outputId":"9dc5b818-736d-4665-96d6-15a4cd7f8dd4"},"outputs":[{"output_type":"stream","name":"stdout","text":["Collecting
+{"nbformat":4,"nbformat_minor":0,"metadata":{"colab":{"provenance":[],"gpuType":"T4"